In [2]:
# ==================================================================================================
# CISLR — L22
# EXPORT FROZEN L21 DEPLOYMENT CLASSIFIER TO OPENVINO
#
# UPDATED VERSION
# ----------------
# FIX:
#   OpenVINO conversion may preserve a dynamic input PartialShape.
#   This version explicitly reshapes the OpenVINO graph to the frozen
#   deployment input:
#
#       [1, 16, 3, 224, 224]
#
#   BEFORE saving / compiling.
#
# IMPORTANT:
#   NO TRAINING
#   NO RETRAINING
#   NO LOCKED TEST INFERENCE
#   NO LOCKED TEST EVALUATION
#   NO QUANTIZATION
#   FP32 OPENVINO EXPORT
#
# L22:
#   L21 frozen PyTorch checkpoint
#       ↓
#   reconstruct exact VideoMAE
#       ↓
#   strict state_dict load
#       ↓
#   logits-only wrapper
#       ↓
#   OpenVINO conversion
#       ↓
#   force STATIC deployment shape [1,16,3,224,224]
#       ↓
#   save XML + BIN
#       ↓
#   reload
#       ↓
#   compile CPU
#       ↓
#   synthetic sanity inference
#       ↓
#   basic PyTorch/OpenVINO comparison
#       ↓
#   hash + freeze L22
#
# L23:
#   detailed PyTorch <-> OpenVINO verification using real VALIDATION clips.
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import gc
import sys
import json
import hashlib
import shutil
import platform
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import openvino as ov

from transformers import VideoMAEForVideoClassification


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def section(title):

    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def save_json(obj, path):

    path = Path(path)

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def safe_port_info(port):

    info = {}

    # ----------------------------------------------------------------------------------------------
    # Names
    # ----------------------------------------------------------------------------------------------

    try:

        info["names"] = sorted(
            list(
                port.get_names()
            )
        )

    except Exception:

        info["names"] = []


    # ----------------------------------------------------------------------------------------------
    # Partial shape
    # ----------------------------------------------------------------------------------------------

    try:

        info["partial_shape"] = str(
            port.partial_shape
        )

    except Exception as e:

        info["partial_shape"] = (
            f"UNAVAILABLE: {e}"
        )


    # ----------------------------------------------------------------------------------------------
    # Static shape only when safe
    # ----------------------------------------------------------------------------------------------

    try:

        partial_shape = (
            port.partial_shape
        )

        if partial_shape.is_static:

            info["static_shape"] = [
                int(x)
                for x
                in partial_shape.to_shape()
            ]

        else:

            info["static_shape"] = None

    except Exception:

        info["static_shape"] = None


    # ----------------------------------------------------------------------------------------------
    # Element type
    # ----------------------------------------------------------------------------------------------

    try:

        info["element_type"] = str(
            port.element_type
        )

    except Exception:

        info["element_type"] = None


    return info


# ==================================================================================================
# 2. HEADER
# ==================================================================================================

section(
    "CISLR — L22 EXPORT DEPLOYMENT MODEL TO OPENVINO"
)


print(
    "Python       :",
    platform.python_version()
)

print(
    "PyTorch      :",
    torch.__version__
)

print(
    "OpenVINO     :",
    ov.__version__
)

print(
    "Transformers :",
    __import__(
        "transformers"
    ).__version__
)


print()
print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


print()
print(
    "Training             : DISABLED"
)

print(
    "Retraining           : DISABLED"
)

print(
    "Locked-test inference: DISABLED"
)

print(
    "Locked-test evaluation: DISABLED"
)

print(
    "Quantization         : DISABLED"
)

print(
    "Export precision     : FP32"
)


# ==================================================================================================
# 3. PATH CONFIGURATION
# ==================================================================================================

section(
    "3. PATH CONFIGURATION"
)


PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)


ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


AUDIT_ROOT = (
    ROOT /
    "audit"
)


# --------------------------------------------------------------------------------------------------
# L21
# --------------------------------------------------------------------------------------------------

L21_ROOT = (
    AUDIT_ROOT /
    "l21_deployment_classifier_freeze"
)


L21_CHECKPOINT = (
    L21_ROOT /
    "model" /
    "l21_frozen_deployment_videomae_10class.pt"
)


L21_CONFIG = (
    L21_ROOT /
    "config" /
    "l21_deployment_config.json"
)


L21_CLASS_MAPPING = (
    L21_ROOT /
    "config" /
    "deployment_class_mapping.csv"
)


L21_VOCABULARY = (
    L21_ROOT /
    "config" /
    "deployment_vocabulary.json"
)


L21_MASTER_FREEZE = (
    L21_ROOT /
    "L21_DEPLOYMENT_CLASSIFIER_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# L22
# --------------------------------------------------------------------------------------------------

L22_ROOT = (
    AUDIT_ROOT /
    "l22_openvino_export"
)


MODEL_DIR = (
    L22_ROOT /
    "model"
)


CONFIG_DIR = (
    L22_ROOT /
    "config"
)


REPORT_DIR = (
    L22_ROOT /
    "reports"
)


for directory in [

    L22_ROOT,
    MODEL_DIR,
    CONFIG_DIR,
    REPORT_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


OPENVINO_XML = (
    MODEL_DIR /
    "l22_cislr_videomae_10class.xml"
)


OPENVINO_BIN = (
    MODEL_DIR /
    "l22_cislr_videomae_10class.bin"
)


L22_CONFIG_FILE = (
    CONFIG_DIR /
    "l22_openvino_export_config.json"
)


L22_REPORT_FILE = (
    REPORT_DIR /
    "l22_openvino_export_report.json"
)


L22_HASH_FILE = (
    REPORT_DIR /
    "l22_artifact_hashes.json"
)


L22_MASTER_FREEZE = (
    L22_ROOT /
    "L22_OPENVINO_EXPORT_MASTER_FREEZE.json"
)


print(
    "L21 checkpoint:"
)

print(
    L21_CHECKPOINT
)


print()
print(
    "L22 output:"
)

print(
    L22_ROOT
)


# ==================================================================================================
# 4. VERIFY REQUIRED L21 ARTIFACTS
# ==================================================================================================

section(
    "4. VERIFY REQUIRED L21 ARTIFACTS"
)


required_files = {

    "L21 checkpoint":
        L21_CHECKPOINT,

    "L21 config":
        L21_CONFIG,

    "L21 class mapping":
        L21_CLASS_MAPPING,

    "L21 vocabulary":
        L21_VOCABULARY,

    "L21 master freeze":
        L21_MASTER_FREEZE
}


for name, path in required_files.items():

    exists = (
        path.exists()
    )

    print(
        f"{name:<25}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


print()
print(
    "[PASS] All required L21 artifacts found."
)


# ==================================================================================================
# 5. VERIFY L21 MASTER FREEZE
# ==================================================================================================

section(
    "5. VERIFY L21 MASTER FREEZE"
)


with open(
    L21_MASTER_FREEZE,
    "r"
) as f:

    l21_master = json.load(f)


print(
    "Stage :",
    l21_master.get(
        "stage"
    )
)

print(
    "Status:",
    l21_master.get(
        "status"
    )
)


if (
    l21_master.get(
        "stage"
    )
    !=
    "L21"
):

    raise RuntimeError(
        "Unexpected L21 stage."
    )


if (
    l21_master.get(
        "status"
    )
    !=
    "FROZEN"
):

    raise RuntimeError(
        "L21 is not frozen."
    )


if bool(
    l21_master.get(
        "training",
        True
    )
):

    raise RuntimeError(
        "L21 unexpectedly reports training."
    )


if bool(
    l21_master.get(
        "locked_test_inference",
        True
    )
):

    raise RuntimeError(
        "L21 unexpectedly reports locked-test inference."
    )


if bool(
    l21_master.get(
        "locked_test_evaluation",
        True
    )
):

    raise RuntimeError(
        "L21 unexpectedly reports locked-test evaluation."
    )


print()
print(
    "[PASS] L21 frozen protocol verified."
)


# ==================================================================================================
# 6. VERIFY L21 CHECKPOINT SHA256
# ==================================================================================================

section(
    "6. VERIFY L21 CHECKPOINT SHA256"
)


EXPECTED_L19_L21_CHECKPOINT_SHA = (
    "3a35abe76cdea634490473eaf83c6121e102658362cfbab19410a4949bb8c91c"
)


actual_l21_checkpoint_sha = (
    sha256_file(
        L21_CHECKPOINT
    )
)


print(
    "Expected:"
)

print(
    EXPECTED_L19_L21_CHECKPOINT_SHA
)


print()
print(
    "Actual:"
)

print(
    actual_l21_checkpoint_sha
)


if (
    actual_l21_checkpoint_sha
    !=
    EXPECTED_L19_L21_CHECKPOINT_SHA
):

    raise RuntimeError(
        "L21 checkpoint SHA256 mismatch."
    )


master_checkpoint_sha = (
    l21_master.get(
        "checkpoint_sha256"
    )
)


if (
    master_checkpoint_sha
    !=
    actual_l21_checkpoint_sha
):

    raise RuntimeError(
        "L21 master-freeze checkpoint SHA "
        "does not match actual checkpoint."
    )


print()
print(
    "[PASS] Exact frozen L21 checkpoint verified."
)


# ==================================================================================================
# 7. LOAD L21 DEPLOYMENT CONFIGURATION
# ==================================================================================================

section(
    "7. LOAD L21 DEPLOYMENT CONFIGURATION"
)


with open(
    L21_CONFIG,
    "r"
) as f:

    deployment_config = json.load(f)


NUM_CLASSES = int(
    deployment_config[
        "num_classes"
    ]
)


MODEL_FRAMES = int(
    deployment_config[
        "num_frames"
    ]
)


INPUT_HEIGHT = int(
    deployment_config[
        "input_resolution"
    ][0]
)


INPUT_WIDTH = int(
    deployment_config[
        "input_resolution"
    ][1]
)


MODEL_NAME = (
    deployment_config[
        "pretrained_backbone"
    ]
)


STATIC_INPUT_SHAPE = [

    1,
    MODEL_FRAMES,
    3,
    INPUT_HEIGHT,
    INPUT_WIDTH

]


EXPECTED_STATIC_INPUT_SHAPE = [

    1,
    16,
    3,
    224,
    224

]


print(
    "Model:",
    MODEL_NAME
)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Frames:",
    MODEL_FRAMES
)

print(
    "Input:",
    f"{INPUT_HEIGHT}x{INPUT_WIDTH}"
)

print(
    "Layout:",
    deployment_config[
        "input_layout"
    ]
)

print(
    "Static deployment shape:",
    STATIC_INPUT_SHAPE
)


if NUM_CLASSES != 10:

    raise RuntimeError(
        "Expected 10 classes."
    )


if MODEL_FRAMES != 16:

    raise RuntimeError(
        "Expected 16 frames."
    )


if (
    INPUT_HEIGHT != 224
    or
    INPUT_WIDTH != 224
):

    raise RuntimeError(
        "Expected 224x224 input."
    )


if (
    deployment_config[
        "input_layout"
    ]
    !=
    "B,T,C,H,W"
):

    raise RuntimeError(
        "Unexpected input layout."
    )


if (
    STATIC_INPUT_SHAPE
    !=
    EXPECTED_STATIC_INPUT_SHAPE
):

    raise RuntimeError(
        "Unexpected deployment input shape."
    )


print()
print(
    "[PASS] L21 deployment configuration verified."
)


# ==================================================================================================
# 8. LOAD DEPLOYMENT VOCABULARY
# ==================================================================================================

section(
    "8. LOAD DEPLOYMENT VOCABULARY"
)


with open(
    L21_VOCABULARY,
    "r"
) as f:

    vocabulary_json = json.load(f)


id_to_gloss = {

    int(k):
        str(v)

    for k, v
    in vocabulary_json.items()
}


EXPECTED_VOCAB = {

    0: "monday",
    1: "interest",
    2: "thursday",
    3: "christmas",
    4: "pencil",
    5: "mother",
    6: "tuesday",
    7: "friday",
    8: "rubber",
    9: "wash"
}


if (
    id_to_gloss
    !=
    EXPECTED_VOCAB
):

    raise RuntimeError(
        "Deployment vocabulary mismatch."
    )


for class_id in range(
    NUM_CLASSES
):

    print(
        f"{class_id:2d} -> "
        f"{id_to_gloss[class_id]}"
    )


print()
print(
    "[PASS] Exact 10-class vocabulary verified."
)


# ==================================================================================================
# 9. LOAD FROZEN CHECKPOINT
# ==================================================================================================

section(
    "9. LOAD FROZEN L21 CHECKPOINT"
)


checkpoint = torch.load(

    L21_CHECKPOINT,

    map_location="cpu",

    weights_only=False
)


if not isinstance(
    checkpoint,
    dict
):

    raise RuntimeError(
        "Unexpected checkpoint structure."
    )


if (
    "model_state_dict"
    not in checkpoint
):

    raise RuntimeError(
        "Expected model_state_dict "
        "not found in checkpoint."
    )


state_dict = (
    checkpoint[
        "model_state_dict"
    ]
)


print(
    "Checkpoint keys:",
    list(
        checkpoint.keys()
    )
)


print(
    "State-dict entries:",
    len(
        state_dict
    )
)


# ==================================================================================================
# 10. RECONSTRUCT EXACT PYTORCH VIDEOMAE
# ==================================================================================================

section(
    "10. RECONSTRUCT EXACT PYTORCH VIDEOMAE"
)


print(
    "Loading architecture:"
)

print(
    MODEL_NAME
)


model = (
    VideoMAEForVideoClassification
    .from_pretrained(

        MODEL_NAME,

        num_labels=
            NUM_CLASSES,

        ignore_mismatched_sizes=
            True,

        label2id={

            id_to_gloss[i]:
                i

            for i in range(
                NUM_CLASSES
            )
        },

        id2label={

            i:
                id_to_gloss[i]

            for i in range(
                NUM_CLASSES
            )
        }
    )
)


print()
print(
    "[PASS] Architecture instantiated."
)


# ==================================================================================================
# 11. STRICTLY LOAD FROZEN L21 WEIGHTS
# ==================================================================================================

section(
    "11. STRICTLY LOAD FROZEN L21 WEIGHTS"
)


load_result = (
    model.load_state_dict(
        state_dict,
        strict=True
    )
)


print(
    "Missing keys   :",
    load_result.missing_keys
)

print(
    "Unexpected keys:",
    load_result.unexpected_keys
)


if len(
    load_result.missing_keys
) != 0:

    raise RuntimeError(
        "Missing model keys."
    )


if len(
    load_result.unexpected_keys
) != 0:

    raise RuntimeError(
        "Unexpected model keys."
    )


print()
print(
    "[PASS] Exact L21 state_dict loaded STRICTLY."
)


# ==================================================================================================
# 12. MODEL PARAMETER VERIFICATION
# ==================================================================================================

section(
    "12. MODEL PARAMETER VERIFICATION"
)


EXPECTED_PARAMETERS = (
    86_234_890
)


total_parameters = sum(

    parameter.numel()

    for parameter
    in model.parameters()
)


print(
    "Expected parameters:",
    f"{EXPECTED_PARAMETERS:,}"
)

print(
    "Actual parameters  :",
    f"{total_parameters:,}"
)


if (
    total_parameters
    !=
    EXPECTED_PARAMETERS
):

    raise RuntimeError(
        "Parameter count mismatch."
    )


for name, parameter in model.named_parameters():

    if not torch.isfinite(
        parameter
    ).all():

        raise RuntimeError(
            f"Non-finite parameter: {name}"
        )


print()
print(
    "[PASS] Model parameter count and values verified."
)


# ==================================================================================================
# 13. SET INFERENCE MODE
# ==================================================================================================

section(
    "13. SET PYTORCH MODEL TO INFERENCE MODE"
)


model = model.cpu()

model.eval()


for parameter in model.parameters():

    parameter.requires_grad = False


trainable_parameters = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    "Trainable parameters:",
    trainable_parameters
)


if (
    trainable_parameters
    !=
    0
):

    raise RuntimeError(
        "Model still contains trainable parameters."
    )


print()
print(
    "[PASS] PyTorch deployment model frozen."
)


# ==================================================================================================
# 14. LOGITS-ONLY EXPORT WRAPPER
# ==================================================================================================

section(
    "14. CREATE LOGITS-ONLY EXPORT WRAPPER"
)


class VideoMAELogitsWrapper(
    nn.Module
):

    def __init__(
        self,
        model
    ):

        super().__init__()

        self.model = model


    def forward(
        self,
        pixel_values
    ):

        outputs = self.model(
            pixel_values=
                pixel_values
        )

        return outputs.logits


wrapper = VideoMAELogitsWrapper(
    model
)


wrapper.eval()


print(
    "Wrapper:",
    wrapper.__class__.__name__
)


print()
print(
    "[PASS] Wrapper returns logits only."
)


# ==================================================================================================
# 15. CREATE DETERMINISTIC SYNTHETIC EXPORT INPUT
# ==================================================================================================

section(
    "15. CREATE DETERMINISTIC EXPORT INPUT"
)


torch.manual_seed(
    42
)


example_input = torch.randn(

    *STATIC_INPUT_SHAPE,

    dtype=torch.float32
)


print(
    "Example input shape:",
    tuple(
        example_input.shape
    )
)

print(
    "dtype:",
    example_input.dtype
)


if (
    list(
        example_input.shape
    )
    !=
    STATIC_INPUT_SHAPE
):

    raise RuntimeError(
        "Unexpected example input shape."
    )


print()
print(
    "[PASS] Synthetic input = [1,16,3,224,224]."
)


# ==================================================================================================
# 16. PYTORCH SANITY INFERENCE
# ==================================================================================================

section(
    "16. PYTORCH SANITY INFERENCE"
)


with torch.inference_mode():

    pytorch_logits = wrapper(
        example_input
    )


print(
    "PyTorch output shape:",
    tuple(
        pytorch_logits.shape
    )
)


if (
    tuple(
        pytorch_logits.shape
    )
    !=
    (
        1,
        NUM_CLASSES
    )
):

    raise RuntimeError(
        "Unexpected PyTorch output shape."
    )


if not torch.isfinite(
    pytorch_logits
).all():

    raise RuntimeError(
        "PyTorch produced non-finite logits."
    )


pytorch_top1 = int(

    torch.argmax(
        pytorch_logits,
        dim=1
    ).item()
)


print(
    "Synthetic PyTorch Top-1:",
    pytorch_top1,
    "->",
    id_to_gloss[
        pytorch_top1
    ]
)


print()
print(
    "[PASS] PyTorch sanity inference succeeded."
)


# ==================================================================================================
# 17. CONVERT PYTORCH -> OPENVINO
# ==================================================================================================

section(
    "17. CONVERT PYTORCH VIDEOMAE TO OPENVINO"
)


print(
    "Starting conversion..."
)


conversion_method = None


try:

    ov_model = ov.convert_model(

        wrapper,

        example_input=
            example_input
    )


    conversion_method = (
        "direct_pytorch"
    )


    print(
        "[PASS] Direct PyTorch conversion succeeded."
    )


except Exception as direct_error:

    print()
    print(
        "[WARNING] Direct conversion failed:"
    )

    print(
        type(
            direct_error
        ).__name__,
        ":",
        direct_error
    )


    print()
    print(
        "Trying TorchScript trace fallback..."
    )


    with torch.inference_mode():

        traced_wrapper = (
            torch.jit.trace(

                wrapper,

                example_input,

                strict=False
            )
        )


    traced_wrapper = (
        torch.jit.freeze(
            traced_wrapper.eval()
        )
    )


    ov_model = ov.convert_model(

        traced_wrapper,

        example_input=
            example_input
    )


    conversion_method = (
        "torchscript_trace_fallback"
    )


    print(
        "[PASS] TorchScript fallback conversion succeeded."
    )


print()
print(
    "Conversion method:",
    conversion_method
)


# ==================================================================================================
# 18. INSPECT RAW CONVERTED OPENVINO SHAPE
# ==================================================================================================

section(
    "18. INSPECT RAW CONVERTED OPENVINO GRAPH"
)


if len(
    ov_model.inputs
) != 1:

    raise RuntimeError(
        f"Expected 1 input, found {len(ov_model.inputs)}."
    )


if len(
    ov_model.outputs
) != 1:

    raise RuntimeError(
        f"Expected 1 output, found {len(ov_model.outputs)}."
    )


raw_input_info = safe_port_info(
    ov_model.input(0)
)


raw_output_info = safe_port_info(
    ov_model.output(0)
)


print(
    "RAW INPUT"
)

print(
    "  names        :",
    raw_input_info[
        "names"
    ]
)

print(
    "  partial shape:",
    raw_input_info[
        "partial_shape"
    ]
)

print(
    "  static shape :",
    raw_input_info[
        "static_shape"
    ]
)

print(
    "  element type :",
    raw_input_info[
        "element_type"
    ]
)


print()
print(
    "RAW OUTPUT"
)

print(
    "  names        :",
    raw_output_info[
        "names"
    ]
)

print(
    "  partial shape:",
    raw_output_info[
        "partial_shape"
    ]
)

print(
    "  static shape :",
    raw_output_info[
        "static_shape"
    ]
)

print(
    "  element type :",
    raw_output_info[
        "element_type"
    ]
)


# ==================================================================================================
# 19. FORCE STATIC DEPLOYMENT INPUT SHAPE
#
# THIS IS THE IMPORTANT FIX FOR YOUR ERROR.
# ==================================================================================================

section(
    "19. FORCE STATIC OPENVINO DEPLOYMENT SHAPE"
)


print(
    "Target static shape:"
)

print(
    STATIC_INPUT_SHAPE
)


# --------------------------------------------------------------------------------------------------
# Use the current input port itself as the reshape key.
#
# This avoids relying on whatever autogenerated name the converter used.
# --------------------------------------------------------------------------------------------------

ov_model.reshape(

    {
        ov_model.input(0):
            ov.PartialShape(
                STATIC_INPUT_SHAPE
            )
    }
)


static_input_info = safe_port_info(
    ov_model.input(0)
)


static_output_info = safe_port_info(
    ov_model.output(0)
)


print()
print(
    "AFTER RESHAPE"
)

print(
    "Input partial shape:",
    static_input_info[
        "partial_shape"
    ]
)

print(
    "Input static shape :",
    static_input_info[
        "static_shape"
    ]
)


print()
print(
    "Output partial shape:",
    static_output_info[
        "partial_shape"
    ]
)

print(
    "Output static shape :",
    static_output_info[
        "static_shape"
    ]
)


if (
    static_input_info[
        "static_shape"
    ]
    !=
    STATIC_INPUT_SHAPE
):

    raise RuntimeError(
        "OpenVINO input could not be frozen to "
        "[1,16,3,224,224]."
    )


if (
    static_output_info[
        "static_shape"
    ]
    is not None
):

    if (
        static_output_info[
            "static_shape"
        ]
        !=
        [
            1,
            NUM_CLASSES
        ]
    ):

        raise RuntimeError(
            "Unexpected static OpenVINO output shape."
        )


print()
print(
    "[PASS] OpenVINO deployment input is now STATIC."
)


# ==================================================================================================
# 20. SET FRIENDLY INPUT / OUTPUT NAMES
# ==================================================================================================

section(
    "20. SET OPENVINO TENSOR NAMES"
)


try:

    ov_model.input(
        0
    ).get_tensor().set_names(
        {
            "pixel_values"
        }
    )

except Exception as error:

    print(
        "[INFO] Input rename warning:",
        error
    )


try:

    ov_model.output(
        0
    ).get_tensor().set_names(
        {
            "logits"
        }
    )

except Exception as error:

    print(
        "[INFO] Output rename warning:",
        error
    )


input_info = safe_port_info(
    ov_model.input(0)
)


output_info = safe_port_info(
    ov_model.output(0)
)


print(
    "Input names :",
    input_info[
        "names"
    ]
)

print(
    "Output names:",
    output_info[
        "names"
    ]
)


# ==================================================================================================
# 21. SAVE FP32 OPENVINO IR
# ==================================================================================================

section(
    "21. SAVE FP32 OPENVINO IR"
)


# --------------------------------------------------------------------------------------------------
# Remove incomplete output from the failed previous L22 run.
# ONLY L22 output files are touched.
# --------------------------------------------------------------------------------------------------

if OPENVINO_XML.exists():

    OPENVINO_XML.unlink()


if OPENVINO_BIN.exists():

    OPENVINO_BIN.unlink()


ov.save_model(

    ov_model,

    OPENVINO_XML,

    compress_to_fp16=False
)


if not OPENVINO_XML.exists():

    raise RuntimeError(
        "OpenVINO XML was not created."
    )


if not OPENVINO_BIN.exists():

    raise RuntimeError(
        "OpenVINO BIN was not created."
    )


print(
    "[SAVED]",
    OPENVINO_XML
)

print(
    "[SAVED]",
    OPENVINO_BIN
)


print()
print(
    "XML size:",
    f"{OPENVINO_XML.stat().st_size / (1024**2):.3f} MB"
)


print(
    "BIN size:",
    f"{OPENVINO_BIN.stat().st_size / (1024**2):.3f} MB"
)


print()
print(
    "[PASS] Static FP32 OpenVINO IR saved."
)


# ==================================================================================================
# 22. RELOAD SAVED OPENVINO MODEL
# ==================================================================================================

section(
    "22. RELOAD SAVED OPENVINO MODEL"
)


core = ov.Core()


print(
    "Available OpenVINO devices:",
    core.available_devices
)


reloaded_ov_model = (
    core.read_model(
        OPENVINO_XML
    )
)


if len(
    reloaded_ov_model.inputs
) != 1:

    raise RuntimeError(
        "Reloaded model input count != 1."
    )


if len(
    reloaded_ov_model.outputs
) != 1:

    raise RuntimeError(
        "Reloaded model output count != 1."
    )


reloaded_input_info = safe_port_info(
    reloaded_ov_model.input(0)
)


reloaded_output_info = safe_port_info(
    reloaded_ov_model.output(0)
)


print(
    "Reloaded input partial shape:",
    reloaded_input_info[
        "partial_shape"
    ]
)

print(
    "Reloaded input static shape :",
    reloaded_input_info[
        "static_shape"
    ]
)


print()
print(
    "Reloaded output partial shape:",
    reloaded_output_info[
        "partial_shape"
    ]
)

print(
    "Reloaded output static shape :",
    reloaded_output_info[
        "static_shape"
    ]
)


if (
    reloaded_input_info[
        "static_shape"
    ]
    !=
    STATIC_INPUT_SHAPE
):

    raise RuntimeError(
        "Saved OpenVINO model did not preserve "
        "static input shape."
    )


if (
    reloaded_output_info[
        "static_shape"
    ]
    !=
    [
        1,
        NUM_CLASSES
    ]
):

    raise RuntimeError(
        "Saved OpenVINO model output shape != [1,10]."
    )


print()
print(
    "[PASS] Saved static OpenVINO model reloaded."
)


# ==================================================================================================
# 23. COMPILE OPENVINO MODEL ON CPU
# ==================================================================================================

section(
    "23. COMPILE OPENVINO MODEL ON CPU"
)


compiled_model = (
    core.compile_model(
        reloaded_ov_model,
        "CPU"
    )
)


compiled_input = (
    compiled_model.input(0)
)


compiled_output = (
    compiled_model.output(0)
)


# --------------------------------------------------------------------------------------------------
# IMPORTANT FIX:
#
# DO NOT blindly call:
#
#     compiled_input.shape
#
# unless the partial shape is confirmed static.
# --------------------------------------------------------------------------------------------------

compiled_input_info = safe_port_info(
    compiled_input
)


compiled_output_info = safe_port_info(
    compiled_output
)


print(
    "Compiled device: CPU"
)


print()
print(
    "COMPILED INPUT"
)

print(
    "  names        :",
    compiled_input_info[
        "names"
    ]
)

print(
    "  partial shape:",
    compiled_input_info[
        "partial_shape"
    ]
)

print(
    "  static shape :",
    compiled_input_info[
        "static_shape"
    ]
)

print(
    "  element type :",
    compiled_input_info[
        "element_type"
    ]
)


print()
print(
    "COMPILED OUTPUT"
)

print(
    "  names        :",
    compiled_output_info[
        "names"
    ]
)

print(
    "  partial shape:",
    compiled_output_info[
        "partial_shape"
    ]
)

print(
    "  static shape :",
    compiled_output_info[
        "static_shape"
    ]
)

print(
    "  element type :",
    compiled_output_info[
        "element_type"
    ]
)


if (
    compiled_input_info[
        "static_shape"
    ]
    !=
    STATIC_INPUT_SHAPE
):

    raise RuntimeError(
        "Compiled OpenVINO input is not "
        "[1,16,3,224,224]."
    )


if (
    compiled_output_info[
        "static_shape"
    ]
    !=
    [
        1,
        NUM_CLASSES
    ]
):

    raise RuntimeError(
        "Compiled OpenVINO output is not [1,10]."
    )


print()
print(
    "[PASS] OpenVINO model compiled with static deployment shapes."
)


# ==================================================================================================
# 24. OPENVINO SYNTHETIC SANITY INFERENCE
# ==================================================================================================

section(
    "24. OPENVINO SANITY INFERENCE"
)


example_numpy = (

    example_input
    .detach()
    .cpu()
    .numpy()
    .astype(
        np.float32,
        copy=False
    )
)


print(
    "Input numpy shape:",
    example_numpy.shape
)

print(
    "Input dtype:",
    example_numpy.dtype
)


ov_result = compiled_model(

    {
        compiled_input:
            example_numpy
    }
)


openvino_logits = np.asarray(

    ov_result[
        compiled_output
    ]
)


print()
print(
    "OpenVINO output shape:",
    openvino_logits.shape
)

print(
    "OpenVINO output dtype:",
    openvino_logits.dtype
)


if (
    tuple(
        openvino_logits.shape
    )
    !=
    (
        1,
        NUM_CLASSES
    )
):

    raise RuntimeError(
        "Unexpected OpenVINO output shape."
    )


if not np.isfinite(
    openvino_logits
).all():

    raise RuntimeError(
        "OpenVINO produced non-finite logits."
    )


openvino_top1 = int(

    np.argmax(
        openvino_logits,
        axis=1
    )[0]
)


print()
print(
    "Synthetic OpenVINO Top-1:",
    openvino_top1,
    "->",
    id_to_gloss[
        openvino_top1
    ]
)


print()
print(
    "[PASS] OpenVINO synthetic inference succeeded."
)


# ==================================================================================================
# 25. BASIC PYTORCH <-> OPENVINO SANITY CHECK
#
# Detailed equivalence testing belongs to L23.
# ==================================================================================================

section(
    "25. BASIC PYTORCH <-> OPENVINO CONSISTENCY"
)


pytorch_numpy = (

    pytorch_logits
    .detach()
    .cpu()
    .numpy()
    .astype(
        np.float32,
        copy=False
    )
)


difference = (

    pytorch_numpy
    -
    openvino_logits
)


max_abs_difference = float(

    np.max(
        np.abs(
            difference
        )
    )
)


mean_abs_difference = float(

    np.mean(
        np.abs(
            difference
        )
    )
)


rmse = float(

    np.sqrt(
        np.mean(
            difference ** 2
        )
    )
)


top1_match = (

    pytorch_top1
    ==
    openvino_top1
)


print(
    "PyTorch Top-1 :",
    pytorch_top1,
    "->",
    id_to_gloss[
        pytorch_top1
    ]
)


print(
    "OpenVINO Top-1:",
    openvino_top1,
    "->",
    id_to_gloss[
        openvino_top1
    ]
)


print()
print(
    "Top-1 match :",
    top1_match
)

print(
    "Max abs diff:",
    max_abs_difference
)

print(
    "Mean abs diff:",
    mean_abs_difference
)

print(
    "RMSE         :",
    rmse
)


if not top1_match:

    raise RuntimeError(
        "PyTorch/OpenVINO synthetic Top-1 mismatch."
    )


print()
print(
    "[PASS] Basic conversion sanity check passed."
)


# ==================================================================================================
# 26. COPY DEPLOYMENT METADATA
# ==================================================================================================

section(
    "26. FREEZE DEPLOYMENT METADATA"
)


L22_VOCABULARY = (
    CONFIG_DIR /
    "deployment_vocabulary.json"
)


L22_CLASS_MAPPING = (
    CONFIG_DIR /
    "deployment_class_mapping.csv"
)


shutil.copy2(

    L21_VOCABULARY,
    L22_VOCABULARY
)


shutil.copy2(

    L21_CLASS_MAPPING,
    L22_CLASS_MAPPING
)


if (
    sha256_file(
        L21_VOCABULARY
    )
    !=
    sha256_file(
        L22_VOCABULARY
    )
):

    raise RuntimeError(
        "Vocabulary copy mismatch."
    )


if (
    sha256_file(
        L21_CLASS_MAPPING
    )
    !=
    sha256_file(
        L22_CLASS_MAPPING
    )
):

    raise RuntimeError(
        "Class mapping copy mismatch."
    )


print(
    "[PASS] Vocabulary copied byte-identically."
)

print(
    "[PASS] Class mapping copied byte-identically."
)


# ==================================================================================================
# 27. SAVE L22 CONFIGURATION
# ==================================================================================================

section(
    "27. SAVE L22 EXPORT CONFIGURATION"
)


export_config = {

    "stage":
        "L22",

    "status":
        "OPENVINO_EXPORTED",

    "source_stage":
        "L21",

    "model_family":
        "VideoMAEForVideoClassification",

    "pretrained_backbone":
        MODEL_NAME,

    "num_classes":
        NUM_CLASSES,

    "num_frames":
        MODEL_FRAMES,

    "input_resolution": [
        INPUT_HEIGHT,
        INPUT_WIDTH
    ],

    "input_layout":
        "B,T,C,H,W",

    "static_input_shape":
        STATIC_INPUT_SHAPE,

    "static_output_shape": [
        1,
        NUM_CLASSES
    ],

    "input_dtype":
        "float32",

    "output":
        "classification_logits",

    "openvino_version":
        ov.__version__,

    "pytorch_version":
        torch.__version__,

    "transformers_version":
        __import__(
            "transformers"
        ).__version__,

    "conversion_method":
        conversion_method,

    "dynamic_input_before_reshape":
        raw_input_info[
            "partial_shape"
        ],

    "input_after_reshape":
        static_input_info[
            "partial_shape"
        ],

    "export_precision":
        "FP32",

    "compress_to_fp16":
        False,

    "quantized":
        False,

    "source_checkpoint":
        str(
            L21_CHECKPOINT
        ),

    "openvino_xml":
        str(
            OPENVINO_XML
        ),

    "openvino_bin":
        str(
            OPENVINO_BIN
        ),

    "preprocessing": {

        "source_cache_frames":
            32,

        "source_cache_resolution": [
            256,
            256
        ],

        "model_frames":
            16,

        "model_resolution": [
            224,
            224
        ],

        "inference_temporal_sampling":
            "deterministic 16-frame sampling from 32-frame cache",

        "spatial_processing":
            "deterministic center crop 256x256 -> 224x224",

        "pixel_scaling":
            "uint8 / 255.0",

        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],

        "normalization_std": [
            0.229,
            0.224,
            0.225
        ],

        "horizontal_flip":
            False,

        "temporal_reversal":
            False
    },

    "vocabulary": {

        str(k):
            v

        for k, v
        in id_to_gloss.items()
    },

    "protocol": {

        "training":
            False,

        "retraining":
            False,

        "weight_modification":
            False,

        "validation_tuning":
            False,

        "locked_test_manifest_loaded":
            False,

        "locked_test_video_loaded":
            False,

        "locked_test_inference":
            False,

        "locked_test_evaluation":
            False,

        "quantization":
            False
    },

    "l22_sanity": {

        "synthetic_input_only":
            True,

        "pytorch_top1":
            pytorch_top1,

        "openvino_top1":
            openvino_top1,

        "top1_match":
            bool(
                top1_match
            ),

        "max_absolute_logit_difference":
            max_abs_difference,

        "mean_absolute_logit_difference":
            mean_abs_difference,

        "rmse":
            rmse
    },

    "next_stage":
        "L23_PYTORCH_OPENVINO_VERIFICATION"
}


save_json(

    export_config,
    L22_CONFIG_FILE
)


print(
    "[SAVED]",
    L22_CONFIG_FILE
)


# ==================================================================================================
# 28. CREATE L22 REPORT
# ==================================================================================================

section(
    "28. CREATE L22 EXPORT REPORT"
)


report = {

    "stage":
        "L22",

    "name":
        "Export selected deployment model to OpenVINO",

    "status":
        "COMPLETE",

    "source": {

        "stage":
            "L21",

        "checkpoint":
            str(
                L21_CHECKPOINT
            ),

        "checkpoint_sha256":
            actual_l21_checkpoint_sha
    },

    "model": {

        "architecture":
            "VideoMAEForVideoClassification",

        "backbone":
            MODEL_NAME,

        "parameters":
            total_parameters,

        "classes":
            NUM_CLASSES,

        "input_shape":
            STATIC_INPUT_SHAPE,

        "output_shape": [
            1,
            NUM_CLASSES
        ]
    },

    "conversion": {

        "method":
            conversion_method,

        "raw_input_partial_shape":
            raw_input_info[
                "partial_shape"
            ],

        "final_static_input_shape":
            STATIC_INPUT_SHAPE,

        "forced_static_shape":
            True
    },

    "openvino": {

        "version":
            ov.__version__,

        "xml":
            str(
                OPENVINO_XML
            ),

        "bin":
            str(
                OPENVINO_BIN
            ),

        "precision":
            "FP32",

        "compiled_device":
            "CPU",

        "compile_success":
            True,

        "sanity_inference_success":
            True
    },

    "sanity_comparison": {

        "input_type":
            "deterministic_synthetic",

        "top1_match":
            bool(
                top1_match
            ),

        "max_absolute_logit_difference":
            max_abs_difference,

        "mean_absolute_logit_difference":
            mean_abs_difference,

        "rmse":
            rmse
    },

    "scientific_integrity": {

        "training":
            False,

        "retraining":
            False,

        "weight_modification":
            False,

        "locked_test_manifest_loaded":
            False,

        "locked_test_video_loaded":
            False,

        "locked_test_inference":
            False,

        "locked_test_evaluation":
            False,

        "test_used_for_export":
            False,

        "validation_used_for_export_tuning":
            False
    },

    "next_stage":
        "L23"
}


save_json(

    report,
    L22_REPORT_FILE
)


print(
    "[SAVED]",
    L22_REPORT_FILE
)


# ==================================================================================================
# 29. HASH L22 ARTIFACTS
# ==================================================================================================

section(
    "29. HASH L22 ARTIFACTS"
)


artifact_paths = {

    "source_l21_checkpoint":
        L21_CHECKPOINT,

    "openvino_xml":
        OPENVINO_XML,

    "openvino_bin":
        OPENVINO_BIN,

    "deployment_vocabulary":
        L22_VOCABULARY,

    "deployment_class_mapping":
        L22_CLASS_MAPPING,

    "export_config":
        L22_CONFIG_FILE,

    "export_report":
        L22_REPORT_FILE
}


artifact_hashes = {}


for name, path in artifact_paths.items():

    digest = (
        sha256_file(
            path
        )
    )


    artifact_hashes[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest,

        "size_bytes":
            int(
                path.stat().st_size
            )
    }


    print(
        f"{name:<30}: "
        f"{digest}"
    )


save_json(

    artifact_hashes,
    L22_HASH_FILE
)


print()
print(
    "[SAVED]",
    L22_HASH_FILE
)


# ==================================================================================================
# 30. FINAL L22 INTEGRITY CHECKS
# ==================================================================================================

section(
    "30. FINAL L22 INTEGRITY CHECKS"
)


checks = {

    "l21_checkpoint_exists":
        L21_CHECKPOINT.exists(),

    "l21_checkpoint_sha_verified":
        (
            actual_l21_checkpoint_sha
            ==
            EXPECTED_L19_L21_CHECKPOINT_SHA
        ),

    "model_parameters_correct":
        (
            total_parameters
            ==
            EXPECTED_PARAMETERS
        ),

    "num_classes_10":
        (
            NUM_CLASSES
            ==
            10
        ),

    "num_frames_16":
        (
            MODEL_FRAMES
            ==
            16
        ),

    "resolution_224x224":
        (
            INPUT_HEIGHT
            ==
            224

            and

            INPUT_WIDTH
            ==
            224
        ),

    "openvino_input_static":
        (
            compiled_input_info[
                "static_shape"
            ]
            ==
            STATIC_INPUT_SHAPE
        ),

    "openvino_output_static":
        (
            compiled_output_info[
                "static_shape"
            ]
            ==
            [
                1,
                NUM_CLASSES
            ]
        ),

    "openvino_xml_exists":
        OPENVINO_XML.exists(),

    "openvino_bin_exists":
        OPENVINO_BIN.exists(),

    "pytorch_output_1x10":
        (
            tuple(
                pytorch_numpy.shape
            )
            ==
            (
                1,
                10
            )
        ),

    "openvino_output_1x10":
        (
            tuple(
                openvino_logits.shape
            )
            ==
            (
                1,
                10
            )
        ),

    "synthetic_top1_match":
        bool(
            top1_match
        ),

    "training_performed":
        False,

    "retraining_performed":
        False,

    "locked_test_inference":
        False,

    "locked_test_evaluation":
        False
}


false_expected = {

    "training_performed",
    "retraining_performed",
    "locked_test_inference",
    "locked_test_evaluation"
}


all_pass = True


for name, value in checks.items():

    if name in false_expected:

        passed = (
            value is False
        )

    else:

        passed = bool(
            value
        )


    print(
        f"{'[PASS]' if passed else '[FAIL]'} "
        f"{name}"
    )


    if not passed:

        all_pass = False


if not all_pass:

    raise RuntimeError(
        "L22 integrity checks failed."
    )


print()
print(
    "[PASS] ALL L22 INTEGRITY CHECKS PASSED."
)


# ==================================================================================================
# 31. CREATE L22 MASTER FREEZE
# ==================================================================================================

section(
    "31. CREATE L22 MASTER FREEZE"
)


master_freeze = {

    "stage":
        "L22",

    "status":
        "COMPLETE_AND_FROZEN",

    "name":
        "OpenVINO deployment model export",

    "source_stage":
        "L21",

    "source_checkpoint": {

        "path":
            str(
                L21_CHECKPOINT
            ),

        "sha256":
            actual_l21_checkpoint_sha
    },

    "openvino_model": {

        "xml":
            str(
                OPENVINO_XML
            ),

        "xml_sha256":
            sha256_file(
                OPENVINO_XML
            ),

        "bin":
            str(
                OPENVINO_BIN
            ),

        "bin_sha256":
            sha256_file(
                OPENVINO_BIN
            ),

        "precision":
            "FP32",

        "static_input_shape":
            STATIC_INPUT_SHAPE,

        "static_output_shape": [
            1,
            NUM_CLASSES
        ]
    },

    "model": {

        "architecture":
            "VideoMAEForVideoClassification",

        "backbone":
            MODEL_NAME,

        "parameters":
            total_parameters,

        "classes":
            NUM_CLASSES
    },

    "conversion": {

        "method":
            conversion_method,

        "raw_input_partial_shape":
            raw_input_info[
                "partial_shape"
            ],

        "static_shape_forced":
            True
    },

    "sanity_verification": {

        "synthetic_input":
            True,

        "pytorch_top1":
            pytorch_top1,

        "openvino_top1":
            openvino_top1,

        "top1_match":
            bool(
                top1_match
            ),

        "max_absolute_logit_difference":
            max_abs_difference,

        "mean_absolute_logit_difference":
            mean_abs_difference,

        "rmse":
            rmse
    },

    "artifacts": {

        "config":
            str(
                L22_CONFIG_FILE
            ),

        "report":
            str(
                L22_REPORT_FILE
            ),

        "hash_manifest":
            str(
                L22_HASH_FILE
            ),

        "vocabulary":
            str(
                L22_VOCABULARY
            ),

        "class_mapping":
            str(
                L22_CLASS_MAPPING
            )
    },

    "protocol": {

        "training":
            False,

        "retraining":
            False,

        "weight_modification":
            False,

        "locked_test_manifest_loaded":
            False,

        "locked_test_video_loaded":
            False,

        "locked_test_inference":
            False,

        "locked_test_evaluation":
            False,

        "quantization":
            False
    },

    "integrity_checks":
        checks,

    "next_stage":
        "L23_PYTORCH_OPENVINO_VERIFICATION"
}


save_json(

    master_freeze,
    L22_MASTER_FREEZE
)


L22_MASTER_SHA = (
    sha256_file(
        L22_MASTER_FREEZE
    )
)


print(
    "[SAVED]"
)

print(
    L22_MASTER_FREEZE
)


print()
print(
    "L22 MASTER SHA256:"
)

print(
    L22_MASTER_SHA
)


# ==================================================================================================
# 32. FINAL REPORT
# ==================================================================================================

section(
    "L22 — OPENVINO EXPORT COMPLETE"
)


print(
    "SOURCE MODEL"
)

print(
    "  Stage        : L21"
)

print(
    "  Architecture : VideoMAEForVideoClassification"
)

print(
    "  Backbone     :",
    MODEL_NAME
)

print(
    "  Parameters   :",
    f"{total_parameters:,}"
)


print()
print(
    "DEPLOYMENT INPUT"
)

print(
    "  Shape        : [1,16,3,224,224]"
)

print(
    "  Layout       : B,T,C,H,W"
)

print(
    "  dtype        : float32"
)

print(
    "  Static       : YES"
)


print()
print(
    "DEPLOYMENT OUTPUT"
)

print(
    "  Shape        : [1,10]"
)

print(
    "  Meaning      : classification logits"
)


print()
print(
    "CONVERSION"
)

print(
    "  Method       :",
    conversion_method
)

print(
    "  Raw OV shape :",
    raw_input_info[
        "partial_shape"
    ]
)

print(
    "  Final shape  :",
    compiled_input_info[
        "partial_shape"
    ]
)


print()
print(
    "OPENVINO"
)

print(
    "  Version      :",
    ov.__version__
)

print(
    "  Precision    : FP32"
)

print(
    "  CPU compile  : PASS"
)

print(
    "  Inference    : PASS"
)


print()
print(
    "BASIC PYTORCH <-> OPENVINO SANITY"
)

print(
    "  PyTorch Top-1 :",
    pytorch_top1,
    "->",
    id_to_gloss[
        pytorch_top1
    ]
)

print(
    "  OpenVINO Top-1:",
    openvino_top1,
    "->",
    id_to_gloss[
        openvino_top1
    ]
)

print(
    "  Top-1 match   :",
    top1_match
)

print(
    "  Max abs diff  :",
    max_abs_difference
)

print(
    "  Mean abs diff :",
    mean_abs_difference
)

print(
    "  RMSE          :",
    rmse
)


print()
print(
    "SCIENTIFIC INTEGRITY"
)

print(
    "  Training              : NO"
)

print(
    "  Retraining            : NO"
)

print(
    "  Weight modification   : NO"
)

print(
    "  Locked test loaded     : NO"
)

print(
    "  Locked test inference  : NO"
)

print(
    "  Locked test evaluation : NO"
)

print(
    "  Validation tuning      : NO"
)

print(
    "  Quantization           : NO"
)


print()
print(
    "OPENVINO XML"
)

print(
    OPENVINO_XML
)


print()
print(
    "OPENVINO BIN"
)

print(
    OPENVINO_BIN
)


print()
print(
    "XML SHA256"
)

print(
    sha256_file(
        OPENVINO_XML
    )
)


print()
print(
    "BIN SHA256"
)

print(
    sha256_file(
        OPENVINO_BIN
    )
)


print()
print(
    "L22 MASTER FREEZE"
)

print(
    L22_MASTER_FREEZE
)


print()
print(
    "L22 MASTER SHA256"
)

print(
    L22_MASTER_SHA
)


print()
print(
    "NEXT"
)

print(
    "L23 — VERIFY PYTORCH <-> OPENVINO PREDICTIONS"
)

print(
    "using real L18B VALIDATION clips only."
)

print(
    "The locked 10-video test remains untouched."
)


print()
print("=" * 120)


# ==================================================================================================
# 33. CLEANUP
# ==================================================================================================

del compiled_model
del reloaded_ov_model
del ov_model
del wrapper
del model
del checkpoint
del state_dict
del example_input


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


print()
print(
    "[PASS] L22 cleanup complete."
)


CISLR — L22 EXPORT DEPLOYMENT MODEL TO OPENVINO
Python       : 3.11.15
PyTorch      : 2.14.0+cu130
OpenVINO     : 2026.3.1-22476-759c5a6ab8c-releases/2026/3
Transformers : 4.57.6

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

Training             : DISABLED
Retraining           : DISABLED
Locked-test inference: DISABLED
Locked-test evaluation: DISABLED
Quantization         : DISABLED
Export precision     : FP32

3. PATH CONFIGURATION
L21 checkpoint:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l21_deployment_classifier_freeze/model/l21_frozen_deployment_videomae_10class.pt

L22 output:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l22_openvino_export

4. VERIFY REQUIRED L21 ARTIFACTS
L21 checkpoint           : PASS
L21 config               : PASS
L21 class mapping        : PASS
L21 vocabulary           : PASS
L21 master freeze        : PASS

[PASS] All required L21 artifacts found.

5. VERIFY L21 MASTER FREEZE
Stage : L21
St

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base-finetuned-kinetics and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[PASS] Architecture instantiated.

11. STRICTLY LOAD FROZEN L21 WEIGHTS
Missing keys   : []
Unexpected keys: []

[PASS] Exact L21 state_dict loaded STRICTLY.

12. MODEL PARAMETER VERIFICATION
Expected parameters: 86,234,890
Actual parameters  : 86,234,890

[PASS] Model parameter count and values verified.

13. SET PYTORCH MODEL TO INFERENCE MODE
Trainable parameters: 0

[PASS] PyTorch deployment model frozen.

14. CREATE LOGITS-ONLY EXPORT WRAPPER
Wrapper: VideoMAELogitsWrapper

[PASS] Wrapper returns logits only.

15. CREATE DETERMINISTIC EXPORT INPUT
Example input shape: (1, 16, 3, 224, 224)
dtype: torch.float32

[PASS] Synthetic input = [1,16,3,224,224].

16. PYTORCH SANITY INFERENCE
PyTorch output shape: (1, 10)
Synthetic PyTorch Top-1: 9 -> wash

[PASS] PyTorch sanity inference succeeded.

17. CONVERT PYTORCH VIDEOMAE TO OPENVINO
Starting conversion...
[PASS] Direct PyTorch conversion succeeded.

Conversion method: direct_pytorch

18. INSPECT RAW CONVERTED OPENVINO GRAPH
RAW INPU

In [3]:
# ==================================================================================================
# CISLR — L23
# VERIFY PYTORCH <-> OPENVINO PREDICTIONS ON REAL L18B VALIDATION CLIPS
#
# INPUTS
# ------
# L21 frozen PyTorch VideoMAE checkpoint
# L22 frozen OpenVINO FP32 IR
# L18B validation manifest ONLY
#
# IMPORTANT
# ---------
# NO TRAINING
# NO RETRAINING
# NO PARAMETER UPDATES
# NO THRESHOLD TUNING
# NO LOCKED TEST MANIFEST LOADING
# NO LOCKED TEST VIDEO LOADING
# NO LOCKED TEST INFERENCE
# NO LOCKED TEST EVALUATION
#
# PURPOSE
# -------
# For every one of the 10 L18B validation samples:
#
#       exact same preprocessed tensor
#                 |
#          -----------------
#          |               |
#       PyTorch         OpenVINO
#          |               |
#        logits          logits
#          |               |
#          ------ compare ---
#
# Measurements:
#   - expected label
#   - PyTorch Top-1
#   - OpenVINO Top-1
#   - PyTorch/OpenVINO Top-1 agreement
#   - Top-5 agreement
#   - probabilities
#   - confidence
#   - max absolute logit error
#   - mean absolute logit error
#   - RMSE
#   - maximum probability error
#   - PyTorch validation accuracy
#   - OpenVINO validation accuracy
#
# L23 DOES NOT USE THE LOCKED TEST SET.
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import gc
import json
import math
import time
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import openvino as ov

from transformers import VideoMAEForVideoClassification


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def section(title):

    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def save_json(obj, path):

    path = Path(path)

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def softmax_numpy(x):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    x = x - np.max(
        x,
        axis=-1,
        keepdims=True
    )

    e = np.exp(x)

    return (
        e /
        np.sum(
            e,
            axis=-1,
            keepdims=True
        )
    )


def find_column(df, candidates):

    for candidate in candidates:

        if candidate in df.columns:

            return candidate

    return None


def safe_static_shape(port):

    try:

        ps = port.partial_shape

        if ps.is_static:

            return [
                int(x)
                for x in ps.to_shape()
            ]

    except Exception:

        pass

    return None


# ==================================================================================================
# 2. HEADER
# ==================================================================================================

section(
    "CISLR — L23 PYTORCH <-> OPENVINO REAL-VALIDATION VERIFICATION"
)


print(
    "Python       :",
    platform.python_version()
)

print(
    "PyTorch      :",
    torch.__version__
)

print(
    "OpenVINO     :",
    ov.__version__
)

print(
    "Transformers :",
    __import__(
        "transformers"
    ).__version__
)


print()
print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


print()
print(
    "Training              : DISABLED"
)

print(
    "Retraining            : DISABLED"
)

print(
    "Validation tuning     : DISABLED"
)

print(
    "Locked-test loading   : DISABLED"
)

print(
    "Locked-test inference : DISABLED"
)

print(
    "Locked-test evaluation: DISABLED"
)


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

section(
    "3. PATH CONFIGURATION"
)


PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)


ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


AUDIT_ROOT = (
    ROOT /
    "audit"
)


# --------------------------------------------------------------------------------------------------
# L18B
# --------------------------------------------------------------------------------------------------

L18B_ROOT = (
    AUDIT_ROOT /
    "l18b_10class_dataset"
)


L18B_MANIFEST_DIR = (
    L18B_ROOT /
    "manifests"
)


VAL_MANIFEST = (
    L18B_MANIFEST_DIR /
    "l18b_validation_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# L21
# --------------------------------------------------------------------------------------------------

L21_ROOT = (
    AUDIT_ROOT /
    "l21_deployment_classifier_freeze"
)


L21_CHECKPOINT = (
    L21_ROOT /
    "model" /
    "l21_frozen_deployment_videomae_10class.pt"
)


L21_CONFIG = (
    L21_ROOT /
    "config" /
    "l21_deployment_config.json"
)


L21_VOCAB = (
    L21_ROOT /
    "config" /
    "deployment_vocabulary.json"
)


L21_MASTER = (
    L21_ROOT /
    "L21_DEPLOYMENT_CLASSIFIER_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# L22
# --------------------------------------------------------------------------------------------------

L22_ROOT = (
    AUDIT_ROOT /
    "l22_openvino_export"
)


OPENVINO_XML = (
    L22_ROOT /
    "model" /
    "l22_cislr_videomae_10class.xml"
)


OPENVINO_BIN = (
    L22_ROOT /
    "model" /
    "l22_cislr_videomae_10class.bin"
)


L22_MASTER = (
    L22_ROOT /
    "L22_OPENVINO_EXPORT_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# L23 OUTPUT
# --------------------------------------------------------------------------------------------------

L23_ROOT = (
    AUDIT_ROOT /
    "l23_pytorch_openvino_verification"
)


REPORT_DIR = (
    L23_ROOT /
    "reports"
)


CONFIG_DIR = (
    L23_ROOT /
    "config"
)


for directory in [

    L23_ROOT,
    REPORT_DIR,
    CONFIG_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


PREDICTIONS_CSV = (
    REPORT_DIR /
    "l23_validation_predictions.csv"
)


SUMMARY_JSON = (
    REPORT_DIR /
    "l23_verification_summary.json"
)


HASH_JSON = (
    REPORT_DIR /
    "l23_artifact_hashes.json"
)


CONFIG_JSON = (
    CONFIG_DIR /
    "l23_verification_config.json"
)


MASTER_FREEZE = (
    L23_ROOT /
    "L23_PYTORCH_OPENVINO_MASTER_FREEZE.json"
)


print(
    "Validation manifest:"
)

print(
    VAL_MANIFEST
)


print()
print(
    "PyTorch checkpoint:"
)

print(
    L21_CHECKPOINT
)


print()
print(
    "OpenVINO XML:"
)

print(
    OPENVINO_XML
)


print()
print(
    "L23 output:"
)

print(
    L23_ROOT
)


# ==================================================================================================
# 4. VERIFY REQUIRED FILES
# ==================================================================================================

section(
    "4. VERIFY REQUIRED ARTIFACTS"
)


required = {

    "validation manifest":
        VAL_MANIFEST,

    "L21 checkpoint":
        L21_CHECKPOINT,

    "L21 config":
        L21_CONFIG,

    "L21 vocabulary":
        L21_VOCAB,

    "L21 master":
        L21_MASTER,

    "L22 XML":
        OPENVINO_XML,

    "L22 BIN":
        OPENVINO_BIN,

    "L22 master":
        L22_MASTER
}


for name, path in required.items():

    exists = path.exists()

    print(
        f"{name:<25}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


print()
print(
    "[PASS] All required L23 inputs found."
)


# ==================================================================================================
# 5. VERIFY FROZEN L21/L22 HASHES
# ==================================================================================================

section(
    "5. VERIFY FROZEN MODEL HASHES"
)


EXPECTED_L21_SHA = (
    "3a35abe76cdea634490473eaf83c6121e102658362cfbab19410a4949bb8c91c"
)


EXPECTED_XML_SHA = (
    "2635acee950a2ebf2286b2892a83bad69082f45721cae3f9c1eeca0e3476c77e"
)


EXPECTED_BIN_SHA = (
    "df19dea76ba03e28820ffd2770982de8c6f5bf6cca93e3ec9c20b12bea3d32fc"
)


actual_l21_sha = sha256_file(
    L21_CHECKPOINT
)


actual_xml_sha = sha256_file(
    OPENVINO_XML
)


actual_bin_sha = sha256_file(
    OPENVINO_BIN
)


print(
    "L21 checkpoint:"
)

print(
    actual_l21_sha
)


print()
print(
    "L22 XML:"
)

print(
    actual_xml_sha
)


print()
print(
    "L22 BIN:"
)

print(
    actual_bin_sha
)


if actual_l21_sha != EXPECTED_L21_SHA:

    raise RuntimeError(
        "Frozen L21 checkpoint hash mismatch."
    )


if actual_xml_sha != EXPECTED_XML_SHA:

    raise RuntimeError(
        "Frozen L22 XML hash mismatch."
    )


if actual_bin_sha != EXPECTED_BIN_SHA:

    raise RuntimeError(
        "Frozen L22 BIN hash mismatch."
    )


print()
print(
    "[PASS] Exact frozen L21/L22 model artifacts verified."
)


# ==================================================================================================
# 6. VERIFY L22 MASTER FREEZE
# ==================================================================================================

section(
    "6. VERIFY L22 MASTER FREEZE"
)


with open(
    L22_MASTER,
    "r"
) as f:

    l22_master = json.load(f)


print(
    "Stage :",
    l22_master.get(
        "stage"
    )
)

print(
    "Status:",
    l22_master.get(
        "status"
    )
)


if l22_master.get("stage") != "L22":

    raise RuntimeError(
        "Unexpected L22 master stage."
    )


if (
    l22_master.get(
        "status"
    )
    !=
    "COMPLETE_AND_FROZEN"
):

    raise RuntimeError(
        "L22 is not frozen."
    )


print()
print(
    "[PASS] L22 master freeze verified."
)


# ==================================================================================================
# 7. LOAD DEPLOYMENT CONFIG + VOCABULARY
# ==================================================================================================

section(
    "7. LOAD FROZEN DEPLOYMENT CONFIGURATION"
)


with open(
    L21_CONFIG,
    "r"
) as f:

    deployment_config = json.load(f)


with open(
    L21_VOCAB,
    "r"
) as f:

    raw_vocab = json.load(f)


id_to_gloss = {

    int(k):
        str(v)

    for k, v
    in raw_vocab.items()
}


EXPECTED_VOCAB = {

    0: "monday",
    1: "interest",
    2: "thursday",
    3: "christmas",
    4: "pencil",
    5: "mother",
    6: "tuesday",
    7: "friday",
    8: "rubber",
    9: "wash"
}


if id_to_gloss != EXPECTED_VOCAB:

    raise RuntimeError(
        "Vocabulary mismatch."
    )


NUM_CLASSES = 10
MODEL_FRAMES = 16
INPUT_SIZE = 224


MODEL_NAME = (
    deployment_config[
        "pretrained_backbone"
    ]
)


print(
    "Backbone:",
    MODEL_NAME
)

print(
    "Classes :",
    NUM_CLASSES
)

print(
    "Frames  :",
    MODEL_FRAMES
)

print(
    "Input   :",
    INPUT_SIZE,
    "x",
    INPUT_SIZE
)


print()
print(
    "Vocabulary:"
)


for k in range(
    NUM_CLASSES
):

    print(
        f"  {k}: {id_to_gloss[k]}"
    )


# ==================================================================================================
# 8. LOAD VALIDATION MANIFEST ONLY
# ==================================================================================================

section(
    "8. LOAD L18B VALIDATION MANIFEST ONLY"
)


val_df = pd.read_csv(
    VAL_MANIFEST
)


print(
    "Validation rows:",
    len(
        val_df
    )
)


print(
    "Columns:"
)

print(
    list(
        val_df.columns
    )
)


if len(
    val_df
) != 10:

    raise RuntimeError(
        f"Expected 10 validation samples, found {len(val_df)}."
    )


# --------------------------------------------------------------------------------------------------
# Identify actual L18B columns.
# --------------------------------------------------------------------------------------------------

UID_COL = find_column(

    val_df,

    [
        "uid",
        "video_uid",
        "video_id",
        "id"
    ]
)


CLASS_COL = find_column(

    val_df,

    [
        "deployment_class_id",
        "class_id",
        "label",
        "target",
        "class"
    ]
)


GLOSS_COL = find_column(

    val_df,

    [
        "gloss",
        "word",
        "sign",
        "class_name"
    ]
)


CACHE_COL = find_column(

    val_df,

    [
        "rgb_cache_path",
        "cache_path",
        "rgb_cache",
        "cached_rgb_path"
    ]
)


VIDEO_COL = find_column(

    val_df,

    [
        "video_path",
        "path",
        "filepath",
        "file_path"
    ]
)


if UID_COL is None:

    raise RuntimeError(
        "Could not identify UID column."
    )


if CLASS_COL is None:

    raise RuntimeError(
        "Could not identify class-ID column."
    )


if GLOSS_COL is None:

    raise RuntimeError(
        "Could not identify gloss column."
    )


if CACHE_COL is None:

    raise RuntimeError(
        "Could not identify RGB-cache column."
    )


print()
print(
    "UID column  :",
    UID_COL
)

print(
    "Class column:",
    CLASS_COL
)

print(
    "Gloss column:",
    GLOSS_COL
)

print(
    "Cache column:",
    CACHE_COL
)

print(
    "Video column:",
    VIDEO_COL
)


# ==================================================================================================
# 9. VERIFY VALIDATION LABELS + CACHE PATHS
# ==================================================================================================

section(
    "9. VERIFY VALIDATION SAMPLES"
)


if val_df[
    UID_COL
].astype(str).duplicated().any():

    raise RuntimeError(
        "Duplicate validation UID detected."
    )


observed_classes = sorted(

    val_df[
        CLASS_COL
    ].astype(int).tolist()
)


if observed_classes != list(
    range(
        NUM_CLASSES
    )
):

    raise RuntimeError(
        "Expected exactly one validation sample "
        "for each deployment class 0..9."
    )


for _, row in val_df.iterrows():

    class_id = int(
        row[
            CLASS_COL
        ]
    )

    gloss = str(
        row[
            GLOSS_COL
        ]
    ).strip().lower()

    if gloss != id_to_gloss[
        class_id
    ]:

        raise RuntimeError(
            f"Class/gloss mismatch: "
            f"{class_id} -> {gloss}"
        )


print(
    "[PASS] Exactly one validation sample per deployment class."
)


# ==================================================================================================
# 10. RESOLVE + AUDIT ALL RGB CACHE FILES
# ==================================================================================================

section(
    "10. AUDIT VALIDATION RGB CACHES"
)


resolved_cache_paths = []


for i, row in val_df.iterrows():

    raw_path = str(
        row[
            CACHE_COL
        ]
    ).strip()


    cache_path = Path(
        raw_path
    ).expanduser()


    # ----------------------------------------------------------------------------------------------
    # If the manifest stored a relative path, try sensible project-relative locations.
    # ----------------------------------------------------------------------------------------------

    if not cache_path.is_absolute():

        candidates = [

            PROJECT_ROOT /
            cache_path,

            ROOT /
            cache_path,

            L18B_ROOT /
            cache_path,

            L18B_MANIFEST_DIR /
            cache_path
        ]


        existing = [

            p
            for p in candidates
            if p.exists()
        ]


        if len(
            existing
        ) == 0:

            raise FileNotFoundError(
                f"Could not resolve cache path: {raw_path}"
            )


        cache_path = existing[0]


    if not cache_path.exists():

        raise FileNotFoundError(
            cache_path
        )


    resolved_cache_paths.append(
        cache_path
    )


    print(
        f"[{i+1:02d}/10] "
        f"{row[UID_COL]} | "
        f"{row[GLOSS_COL]} | "
        f"{cache_path.name}"
    )


val_df = val_df.copy()


val_df[
    "_resolved_cache_path"
] = [

    str(p)

    for p in resolved_cache_paths
]


print()
print(
    "[PASS] All 10 validation RGB caches found."
)


# ==================================================================================================
# 11. EXACT L19 DETERMINISTIC PREPROCESSING
# ==================================================================================================

section(
    "11. DEFINE EXACT L19 INFERENCE PREPROCESSING"
)


IMAGENET_MEAN = np.array(

    [
        0.485,
        0.456,
        0.406
    ],

    dtype=np.float32
)


IMAGENET_STD = np.array(

    [
        0.229,
        0.224,
        0.225
    ],

    dtype=np.float32
)


def load_cache_array(cache_path):

    """
    Load the frozen L18B RGB cache.

    Supports the likely L18B cache formats:
      - .npy
      - .npz

    Returns:
      ndarray expected to represent a video clip.
    """

    cache_path = Path(
        cache_path
    )


    if cache_path.suffix.lower() == ".npy":

        arr = np.load(
            cache_path,
            allow_pickle=False
        )


    elif cache_path.suffix.lower() == ".npz":

        data = np.load(
            cache_path,
            allow_pickle=False
        )


        keys = list(
            data.files
        )


        if len(
            keys
        ) == 0:

            raise RuntimeError(
                f"Empty NPZ cache: {cache_path}"
            )


        # Prefer common names.

        preferred = [

            "frames",
            "rgb",
            "video",
            "clip",
            "arr_0"
        ]


        selected_key = None


        for key in preferred:

            if key in keys:

                selected_key = key
                break


        if selected_key is None:

            if len(keys) == 1:

                selected_key = keys[0]

            else:

                raise RuntimeError(
                    f"Ambiguous NPZ cache keys "
                    f"{keys} for {cache_path}"
                )


        arr = data[
            selected_key
        ]


    else:

        raise RuntimeError(
            f"Unsupported cache format: "
            f"{cache_path.suffix}"
        )


    return np.asarray(
        arr
    )


def standardize_cache_layout(arr):

    """
    Convert cache to T,H,W,C.

    Accepted layouts include:
      T,H,W,C
      T,C,H,W

    We do NOT silently accept arbitrary layouts.
    """

    arr = np.asarray(
        arr
    )


    if arr.ndim != 4:

        raise RuntimeError(
            f"Expected 4-D RGB cache, got {arr.shape}"
        )


    # T,H,W,C

    if arr.shape[-1] == 3:

        frames = arr


    # T,C,H,W

    elif arr.shape[1] == 3:

        frames = np.transpose(

            arr,

            (
                0,
                2,
                3,
                1
            )
        )


    else:

        raise RuntimeError(
            f"Cannot identify RGB channel dimension "
            f"for cache shape {arr.shape}"
        )


    if frames.shape[-1] != 3:

        raise RuntimeError(
            "RGB cache does not contain 3 channels."
        )


    return frames


def deterministic_temporal_sample(
    frames,
    target_frames=16
):

    """
    L19 deterministic validation/inference sampling.

    Select target_frames uniformly across the available cache.
    """

    total = int(
        frames.shape[0]
    )


    if total < 1:

        raise RuntimeError(
            "Empty RGB cache."
        )


    indices = np.linspace(

        0,
        total - 1,
        target_frames

    )


    indices = np.rint(
        indices
    ).astype(
        np.int64
    )


    indices = np.clip(

        indices,
        0,
        total - 1
    )


    return (
        frames[
            indices
        ],
        indices
    )


def center_crop_224(frames):

    """
    Deterministic spatial center crop.

    Expected L18B cache resolution is 256x256.
    """

    h = int(
        frames.shape[1]
    )

    w = int(
        frames.shape[2]
    )


    if (
        h < INPUT_SIZE
        or
        w < INPUT_SIZE
    ):

        raise RuntimeError(
            f"Cache resolution {h}x{w} "
            f"is smaller than 224x224."
        )


    top = (
        h - INPUT_SIZE
    ) // 2


    left = (
        w - INPUT_SIZE
    ) // 2


    cropped = frames[

        :,

        top:
        top + INPUT_SIZE,

        left:
        left + INPUT_SIZE,

        :
    ]


    if cropped.shape[
        1:
    ] != (
        INPUT_SIZE,
        INPUT_SIZE,
        3
    ):

        raise RuntimeError(
            f"Unexpected crop shape: "
            f"{cropped.shape}"
        )


    return cropped


def preprocess_l19(cache_path):

    """
    Exact deterministic deployment tensor:

      cache
        -> T,H,W,C
        -> deterministic 16-frame sampling
        -> center crop to 224x224
        -> float32 / 255
        -> ImageNet normalization
        -> T,C,H,W
        -> B,T,C,H,W

    Returns:
      torch tensor [1,16,3,224,224]
      selected temporal indices
      original cache shape
    """

    raw = load_cache_array(
        cache_path
    )


    original_shape = tuple(
        raw.shape
    )


    frames = standardize_cache_layout(
        raw
    )


    frames, indices = (
        deterministic_temporal_sample(

            frames,

            target_frames=
                MODEL_FRAMES
        )
    )


    frames = center_crop_224(
        frames
    )


    frames = frames.astype(
        np.float32
    )


    # ----------------------------------------------------------------------------------------------
    # L18B RGB cache was stored as image intensity values.
    # Avoid dividing twice if data is already [0,1].
    # ----------------------------------------------------------------------------------------------

    max_value = float(
        np.max(
            frames
        )
    )


    min_value = float(
        np.min(
            frames
        )
    )


    if (
        min_value >= 0.0
        and
        max_value <= 1.0 + 1e-6
    ):

        scaled = frames


    elif (
        min_value >= 0.0
        and
        max_value <= 255.0 + 1e-6
    ):

        scaled = (
            frames /
            255.0
        )


    else:

        raise RuntimeError(
            f"Unexpected RGB cache range "
            f"[{min_value}, {max_value}] "
            f"for {cache_path}"
        )


    normalized = (

        scaled -
        IMAGENET_MEAN.reshape(
            1,
            1,
            1,
            3
        )

    ) / IMAGENET_STD.reshape(

        1,
        1,
        1,
        3
    )


    tensor_np = np.transpose(

        normalized,

        (
            0,
            3,
            1,
            2
        )
    )


    tensor_np = np.ascontiguousarray(
        tensor_np,
        dtype=np.float32
    )


    tensor_np = tensor_np[
        None,
        ...
    ]


    expected_shape = (

        1,
        MODEL_FRAMES,
        3,
        INPUT_SIZE,
        INPUT_SIZE
    )


    if tensor_np.shape != expected_shape:

        raise RuntimeError(
            f"Preprocessed tensor shape "
            f"{tensor_np.shape} != {expected_shape}"
        )


    if not np.isfinite(
        tensor_np
    ).all():

        raise RuntimeError(
            "Preprocessed tensor contains "
            "NaN/Inf."
        )


    tensor_torch = torch.from_numpy(
        tensor_np
    )


    return (
        tensor_torch,
        indices,
        original_shape
    )


print(
    "[PASS] Deterministic L19 preprocessing defined."
)


# ==================================================================================================
# 12. PREPROCESSING AUDIT ON ALL 10 VALIDATION CACHES
# ==================================================================================================

section(
    "12. AUDIT PREPROCESSING ON ALL VALIDATION SAMPLES"
)


preprocessing_audit = []


for i, row in val_df.iterrows():

    cache_path = Path(
        row[
            "_resolved_cache_path"
        ]
    )


    tensor, indices, raw_shape = (
        preprocess_l19(
            cache_path
        )
    )


    preprocessing_audit.append({

        "uid":
            str(
                row[
                    UID_COL
                ]
            ),

        "gloss":
            str(
                row[
                    GLOSS_COL
                ]
            ),

        "raw_shape":
            list(
                raw_shape
            ),

        "sampled_indices":
            [
                int(x)
                for x in indices
            ],

        "tensor_shape":
            list(
                tensor.shape
            ),

        "tensor_min":
            float(
                tensor.min().item()
            ),

        "tensor_max":
            float(
                tensor.max().item()
            )
    })


    print(
        f"[{i+1:02d}/10] "
        f"{row[GLOSS_COL]:<12} "
        f"raw={raw_shape} "
        f"-> tensor={tuple(tensor.shape)}"
    )


    del tensor


print()
print(
    "[PASS] All validation caches preprocess successfully."
)


# ==================================================================================================
# 13. RECONSTRUCT FROZEN PYTORCH MODEL
# ==================================================================================================

section(
    "13. RECONSTRUCT FROZEN PYTORCH MODEL"
)


checkpoint = torch.load(

    L21_CHECKPOINT,

    map_location="cpu",

    weights_only=False
)


if (
    "model_state_dict"
    not in checkpoint
):

    raise RuntimeError(
        "model_state_dict not found."
    )


state_dict = checkpoint[
    "model_state_dict"
]


model = (
    VideoMAEForVideoClassification
    .from_pretrained(

        MODEL_NAME,

        num_labels=
            NUM_CLASSES,

        ignore_mismatched_sizes=
            True,

        label2id={

            id_to_gloss[i]:
                i

            for i in range(
                NUM_CLASSES
            )
        },

        id2label={

            i:
                id_to_gloss[i]

            for i in range(
                NUM_CLASSES
            )
        }
    )
)


load_result = model.load_state_dict(

    state_dict,

    strict=True
)


if (
    len(
        load_result.missing_keys
    )
    != 0
):

    raise RuntimeError(
        "Missing PyTorch state keys."
    )


if (
    len(
        load_result.unexpected_keys
    )
    != 0
):

    raise RuntimeError(
        "Unexpected PyTorch state keys."
    )


model.eval()


for p in model.parameters():

    p.requires_grad = False


TOTAL_PARAMETERS = sum(

    p.numel()

    for p in model.parameters()
)


print(
    "Parameters:",
    f"{TOTAL_PARAMETERS:,}"
)


if TOTAL_PARAMETERS != 86_234_890:

    raise RuntimeError(
        "PyTorch parameter count mismatch."
    )


print()
print(
    "[PASS] Frozen L21 PyTorch model reconstructed."
)


# ==================================================================================================
# 14. LOAD + COMPILE FROZEN OPENVINO MODEL
# ==================================================================================================

section(
    "14. LOAD FROZEN OPENVINO MODEL"
)


core = ov.Core()


ov_model = core.read_model(
    OPENVINO_XML
)


input_shape = safe_static_shape(
    ov_model.input(0)
)


output_shape = safe_static_shape(
    ov_model.output(0)
)


print(
    "Input shape :",
    input_shape
)

print(
    "Output shape:",
    output_shape
)


EXPECTED_INPUT = [
    1,
    16,
    3,
    224,
    224
]


EXPECTED_OUTPUT = [
    1,
    10
]


if input_shape != EXPECTED_INPUT:

    raise RuntimeError(
        f"Unexpected OpenVINO input shape: {input_shape}"
    )


if output_shape != EXPECTED_OUTPUT:

    raise RuntimeError(
        f"Unexpected OpenVINO output shape: {output_shape}"
    )


compiled_model = core.compile_model(

    ov_model,

    "CPU"
)


ov_input = compiled_model.input(0)

ov_output = compiled_model.output(0)


print(
    "Device:",
    "CPU"
)


print()
print(
    "[PASS] Frozen L22 OpenVINO model compiled."
)


# ==================================================================================================
# 15. VERIFY SAME TENSOR IS USED BY BOTH ENGINES
# ==================================================================================================

section(
    "15. BEGIN REAL VALIDATION EQUIVALENCE TEST"
)


print(
    "Samples:",
    len(
        val_df
    )
)


print(
    "IMPORTANT:"
)

print(
    "The SAME float32 tensor is supplied to PyTorch and OpenVINO."
)

print(
    "No independent preprocessing path is used."
)


# ==================================================================================================
# 16. RUN ALL 10 VALIDATION SAMPLES
# ==================================================================================================

results = []


pytorch_correct = 0

openvino_correct = 0

engine_top1_agreement = 0

engine_top5_set_agreement = 0


all_logit_max_errors = []

all_logit_mean_errors = []

all_rmse = []

all_probability_max_errors = []


for sample_index, row in val_df.reset_index(
    drop=True
).iterrows():

    print()
    print(
        "-" * 120
    )


    uid = str(
        row[
            UID_COL
        ]
    )


    expected_id = int(
        row[
            CLASS_COL
        ]
    )


    expected_gloss = str(
        row[
            GLOSS_COL
        ]
    )


    cache_path = Path(
        row[
            "_resolved_cache_path"
        ]
    )


    print(
        f"[{sample_index + 1:02d}/10] "
        f"UID={uid}"
    )

    print(
        "Expected:",
        expected_id,
        "->",
        expected_gloss
    )


    # ----------------------------------------------------------------------------------------------
    # Exact single preprocessing operation.
    # ----------------------------------------------------------------------------------------------

    tensor_torch, temporal_indices, raw_shape = (
        preprocess_l19(
            cache_path
        )
    )


    tensor_numpy = (

        tensor_torch
        .numpy()
        .astype(
            np.float32,
            copy=False
        )
    )


    # ==============================================================================================
    # PyTorch
    # ==============================================================================================

    pt_start = time.perf_counter()


    with torch.inference_mode():

        pt_output = model(
            pixel_values=
                tensor_torch
        )

        pt_logits_tensor = (
            pt_output.logits
        )


    pt_elapsed_ms = (

        time.perf_counter()
        -
        pt_start

    ) * 1000.0


    pt_logits = (

        pt_logits_tensor
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32,
            copy=False
        )
    )


    # ==============================================================================================
    # OpenVINO
    # ==============================================================================================

    ov_start = time.perf_counter()


    ov_result = compiled_model(

        {
            ov_input:
                tensor_numpy
        }
    )


    ov_elapsed_ms = (

        time.perf_counter()
        -
        ov_start

    ) * 1000.0


    ov_logits = np.asarray(

        ov_result[
            ov_output
        ],

        dtype=np.float32
    )


    # ==============================================================================================
    # Structural checks
    # ==============================================================================================

    if pt_logits.shape != (
        1,
        NUM_CLASSES
    ):

        raise RuntimeError(
            f"Bad PyTorch output shape: "
            f"{pt_logits.shape}"
        )


    if ov_logits.shape != (
        1,
        NUM_CLASSES
    ):

        raise RuntimeError(
            f"Bad OpenVINO output shape: "
            f"{ov_logits.shape}"
        )


    if not np.isfinite(
        pt_logits
    ).all():

        raise RuntimeError(
            f"Non-finite PyTorch logits "
            f"for UID={uid}"
        )


    if not np.isfinite(
        ov_logits
    ).all():

        raise RuntimeError(
            f"Non-finite OpenVINO logits "
            f"for UID={uid}"
        )


    # ==============================================================================================
    # Probabilities
    # ==============================================================================================

    pt_probs = softmax_numpy(
        pt_logits
    )


    ov_probs = softmax_numpy(
        ov_logits
    )


    # ==============================================================================================
    # Top-1
    # ==============================================================================================

    pt_top1 = int(

        np.argmax(
            pt_logits[
                0
            ]
        )
    )


    ov_top1 = int(

        np.argmax(
            ov_logits[
                0
            ]
        )
    )


    pt_confidence = float(
        pt_probs[
            0,
            pt_top1
        ]
    )


    ov_confidence = float(
        ov_probs[
            0,
            ov_top1
        ]
    )


    # ==============================================================================================
    # Top-5
    # ==============================================================================================

    pt_top5 = (

        np.argsort(
            -pt_logits[
                0
            ]
        )[
            :5
        ]
        .astype(int)
        .tolist()
    )


    ov_top5 = (

        np.argsort(
            -ov_logits[
                0
            ]
        )[
            :5
        ]
        .astype(int)
        .tolist()
    )


    # ==============================================================================================
    # Numerical differences
    # ==============================================================================================

    logit_difference = (

        pt_logits.astype(
            np.float64
        )
        -
        ov_logits.astype(
            np.float64
        )
    )


    max_abs_logit_error = float(

        np.max(
            np.abs(
                logit_difference
            )
        )
    )


    mean_abs_logit_error = float(

        np.mean(
            np.abs(
                logit_difference
            )
        )
    )


    rmse = float(

        np.sqrt(
            np.mean(
                logit_difference ** 2
            )
        )
    )


    max_probability_error = float(

        np.max(
            np.abs(
                pt_probs -
                ov_probs
            )
        )
    )


    # ==============================================================================================
    # Agreement
    # ==============================================================================================

    top1_match = (
        pt_top1
        ==
        ov_top1
    )


    top5_exact_order_match = (
        pt_top5
        ==
        ov_top5
    )


    top5_set_match = (
        set(
            pt_top5
        )
        ==
        set(
            ov_top5
        )
    )


    pt_is_correct = (
        pt_top1
        ==
        expected_id
    )


    ov_is_correct = (
        ov_top1
        ==
        expected_id
    )


    if pt_is_correct:

        pytorch_correct += 1


    if ov_is_correct:

        openvino_correct += 1


    if top1_match:

        engine_top1_agreement += 1


    if top5_set_match:

        engine_top5_set_agreement += 1


    all_logit_max_errors.append(
        max_abs_logit_error
    )


    all_logit_mean_errors.append(
        mean_abs_logit_error
    )


    all_rmse.append(
        rmse
    )


    all_probability_max_errors.append(
        max_probability_error
    )


    # ==============================================================================================
    # Print sample result
    # ==============================================================================================

    print(
        "PyTorch :",
        pt_top1,
        "->",
        id_to_gloss[
            pt_top1
        ],
        f"| conf={pt_confidence:.8f}"
    )


    print(
        "OpenVINO:",
        ov_top1,
        "->",
        id_to_gloss[
            ov_top1
        ],
        f"| conf={ov_confidence:.8f}"
    )


    print(
        "Correct PT/OV:",
        pt_is_correct,
        "/",
        ov_is_correct
    )


    print(
        "Top-1 engine match:",
        top1_match
    )


    print(
        "Top-5 exact order match:",
        top5_exact_order_match
    )


    print(
        "Top-5 set match:",
        top5_set_match
    )


    print(
        "Max abs logit error:",
        f"{max_abs_logit_error:.12e}"
    )


    print(
        "Max probability error:",
        f"{max_probability_error:.12e}"
    )


    # ==============================================================================================
    # Store
    # ==============================================================================================

    results.append({

        "sample_index":
            int(
                sample_index
            ),

        "uid":
            uid,

        "expected_class_id":
            expected_id,

        "expected_gloss":
            expected_gloss,

        "cache_path":
            str(
                cache_path
            ),

        "raw_cache_shape":
            str(
                tuple(
                    raw_shape
                )
            ),

        "temporal_indices":
            ",".join(
                str(
                    int(x)
                )
                for x in temporal_indices
            ),

        "pytorch_top1_id":
            pt_top1,

        "pytorch_top1_gloss":
            id_to_gloss[
                pt_top1
            ],

        "pytorch_confidence":
            pt_confidence,

        "pytorch_correct":
            bool(
                pt_is_correct
            ),

        "openvino_top1_id":
            ov_top1,

        "openvino_top1_gloss":
            id_to_gloss[
                ov_top1
            ],

        "openvino_confidence":
            ov_confidence,

        "openvino_correct":
            bool(
                ov_is_correct
            ),

        "top1_engine_match":
            bool(
                top1_match
            ),

        "pytorch_top5_ids":
            ",".join(
                map(
                    str,
                    pt_top5
                )
            ),

        "pytorch_top5_glosses":
            ",".join(
                id_to_gloss[x]
                for x in pt_top5
            ),

        "openvino_top5_ids":
            ",".join(
                map(
                    str,
                    ov_top5
                )
            ),

        "openvino_top5_glosses":
            ",".join(
                id_to_gloss[x]
                for x in ov_top5
            ),

        "top5_exact_order_match":
            bool(
                top5_exact_order_match
            ),

        "top5_set_match":
            bool(
                top5_set_match
            ),

        "max_abs_logit_error":
            max_abs_logit_error,

        "mean_abs_logit_error":
            mean_abs_logit_error,

        "rmse":
            rmse,

        "max_probability_error":
            max_probability_error,

        "pytorch_inference_ms":
            float(
                pt_elapsed_ms
            ),

        "openvino_inference_ms":
            float(
                ov_elapsed_ms
            )
    })


    del tensor_torch
    del tensor_numpy
    del pt_output
    del pt_logits_tensor


# ==================================================================================================
# 17. SAVE PER-SAMPLE RESULTS
# ==================================================================================================

section(
    "17. SAVE PER-SAMPLE VERIFICATION RESULTS"
)


results_df = pd.DataFrame(
    results
)


results_df.to_csv(

    PREDICTIONS_CSV,

    index=False
)


print(
    "[SAVED]"
)

print(
    PREDICTIONS_CSV
)


# ==================================================================================================
# 18. CALCULATE OVERALL VALIDATION METRICS
# ==================================================================================================

section(
    "18. OVERALL L23 RESULTS"
)


N = len(
    results_df
)


pytorch_accuracy = (

    pytorch_correct /
    N
)


openvino_accuracy = (

    openvino_correct /
    N
)


top1_agreement_rate = (

    engine_top1_agreement /
    N
)


top5_set_agreement_rate = (

    engine_top5_set_agreement /
    N
)


global_max_logit_error = float(

    np.max(
        all_logit_max_errors
    )
)


mean_sample_max_logit_error = float(

    np.mean(
        all_logit_max_errors
    )
)


mean_logit_error = float(

    np.mean(
        all_logit_mean_errors
    )
)


mean_rmse = float(

    np.mean(
        all_rmse
    )
)


global_max_probability_error = float(

    np.max(
        all_probability_max_errors
    )
)


mean_pt_ms = float(

    results_df[
        "pytorch_inference_ms"
    ].mean()
)


mean_ov_ms = float(

    results_df[
        "openvino_inference_ms"
    ].mean()
)


print(
    "Validation samples:",
    N
)


print()
print(
    "PyTorch accuracy:"
)

print(
    f"  {pytorch_correct}/{N} "
    f"= {100*pytorch_accuracy:.3f}%"
)


print()
print(
    "OpenVINO accuracy:"
)

print(
    f"  {openvino_correct}/{N} "
    f"= {100*openvino_accuracy:.3f}%"
)


print()
print(
    "ENGINE AGREEMENT"
)

print(
    f"  Top-1 agreement : "
    f"{engine_top1_agreement}/{N} "
    f"= {100*top1_agreement_rate:.3f}%"
)


print(
    f"  Top-5 set match : "
    f"{engine_top5_set_agreement}/{N} "
    f"= {100*top5_set_agreement_rate:.3f}%"
)


print()
print(
    "NUMERICAL DIFFERENCE"
)

print(
    "  Global max abs logit error :",
    f"{global_max_logit_error:.12e}"
)

print(
    "  Mean sample max logit error:",
    f"{mean_sample_max_logit_error:.12e}"
)

print(
    "  Mean absolute logit error  :",
    f"{mean_logit_error:.12e}"
)

print(
    "  Mean RMSE                  :",
    f"{mean_rmse:.12e}"
)

print(
    "  Global max probability err :",
    f"{global_max_probability_error:.12e}"
)


print()
print(
    "RAW SINGLE-SAMPLE TIMING"
)

print(
    f"  PyTorch mean : {mean_pt_ms:.3f} ms"
)

print(
    f"  OpenVINO mean: {mean_ov_ms:.3f} ms"
)

print()
print(
    "NOTE: These 10-sample timings are diagnostic only."
)

print(
    "Formal latency/FPS benchmarking belongs to L32."
)


# ==================================================================================================
# 19. VERIFICATION CRITERIA
# ==================================================================================================

section(
    "19. L23 EQUIVALENCE CRITERIA"
)


# --------------------------------------------------------------------------------------------------
# These criteria evaluate ENGINE EQUIVALENCE, not recognition accuracy.
#
# Accuracy is reported separately and is never altered.
# --------------------------------------------------------------------------------------------------

TOP1_AGREEMENT_REQUIRED = 1.0


MAX_LOGIT_ERROR_LIMIT = (
    1e-4
)


MAX_PROBABILITY_ERROR_LIMIT = (
    1e-5
)


top1_equivalence_pass = (

    top1_agreement_rate
    ==
    TOP1_AGREEMENT_REQUIRED
)


logit_equivalence_pass = (

    global_max_logit_error
    <=
    MAX_LOGIT_ERROR_LIMIT
)


probability_equivalence_pass = (

    global_max_probability_error
    <=
    MAX_PROBABILITY_ERROR_LIMIT
)


accuracy_preserved = (

    pytorch_correct
    ==
    openvino_correct
)


all_equivalence_pass = (

    top1_equivalence_pass

    and

    logit_equivalence_pass

    and

    probability_equivalence_pass

    and

    accuracy_preserved
)


criteria = {

    "top1_agreement_100_percent":
        bool(
            top1_equivalence_pass
        ),

    "global_max_logit_error_le_1e-4":
        bool(
            logit_equivalence_pass
        ),

    "global_max_probability_error_le_1e-5":
        bool(
            probability_equivalence_pass
        ),

    "validation_correct_count_preserved":
        bool(
            accuracy_preserved
        )
}


for name, passed in criteria.items():

    print(
        f"{'[PASS]' if passed else '[FAIL]'} "
        f"{name}"
    )


print()


if all_equivalence_pass:

    print(
        "[PASS] PYTORCH <-> OPENVINO "
        "REAL-VALIDATION EQUIVALENCE VERIFIED."
    )

else:

    print(
        "[FAIL] Full engine-equivalence criteria "
        "were not satisfied."
    )


# ==================================================================================================
# 20. PRINT SAMPLE-BY-SAMPLE TABLE
# ==================================================================================================

section(
    "20. SAMPLE-BY-SAMPLE SUMMARY"
)


display_columns = [

    "expected_gloss",

    "pytorch_top1_gloss",

    "openvino_top1_gloss",

    "pytorch_confidence",

    "openvino_confidence",

    "top1_engine_match",

    "max_abs_logit_error"
]


print(

    results_df[
        display_columns
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 21. SAVE L23 CONFIG
# ==================================================================================================

section(
    "21. SAVE L23 CONFIGURATION"
)


l23_config = {

    "stage":
        "L23",

    "name":
        "PyTorch OpenVINO prediction verification",

    "evaluation_population":
        "L18B validation only",

    "num_samples":
        N,

    "locked_test_used":
        False,

    "source_pytorch_checkpoint":
        str(
            L21_CHECKPOINT
        ),

    "source_openvino_xml":
        str(
            OPENVINO_XML
        ),

    "source_openvino_bin":
        str(
            OPENVINO_BIN
        ),

    "openvino_device":
        "CPU",

    "input_shape": [
        1,
        16,
        3,
        224,
        224
    ],

    "output_shape": [
        1,
        10
    ],

    "preprocessing": {

        "cache":
            "L18B RGB cache",

        "temporal_sampling":
            "deterministic uniform 16-frame sampling",

        "crop":
            "center 224x224",

        "scaling":
            "0..255 -> 0..1",

        "mean": [
            0.485,
            0.456,
            0.406
        ],

        "std": [
            0.229,
            0.224,
            0.225
        ],

        "same_tensor_to_both_engines":
            True
    },

    "equivalence_criteria": {

        "top1_agreement_required":
            1.0,

        "max_absolute_logit_error":
            MAX_LOGIT_ERROR_LIMIT,

        "max_probability_error":
            MAX_PROBABILITY_ERROR_LIMIT,

        "correct_count_must_match":
            True
    },

    "scientific_integrity": {

        "training":
            False,

        "retraining":
            False,

        "threshold_tuning":
            False,

        "locked_test_manifest_loaded":
            False,

        "locked_test_video_loaded":
            False,

        "locked_test_inference":
            False,

        "locked_test_evaluation":
            False
    }
}


save_json(

    l23_config,
    CONFIG_JSON
)


print(
    "[SAVED]"
)

print(
    CONFIG_JSON
)


# ==================================================================================================
# 22. SAVE L23 SUMMARY
# ==================================================================================================

section(
    "22. SAVE L23 VERIFICATION SUMMARY"
)


summary = {

    "stage":
        "L23",

    "status":
        (
            "PASS"
            if all_equivalence_pass
            else
            "FAIL"
        ),

    "samples":
        N,

    "pytorch": {

        "correct":
            int(
                pytorch_correct
            ),

        "accuracy":
            float(
                pytorch_accuracy
            ),

        "accuracy_percent":
            float(
                100 *
                pytorch_accuracy
            )
    },

    "openvino": {

        "correct":
            int(
                openvino_correct
            ),

        "accuracy":
            float(
                openvino_accuracy
            ),

        "accuracy_percent":
            float(
                100 *
                openvino_accuracy
            )
    },

    "engine_agreement": {

        "top1_matches":
            int(
                engine_top1_agreement
            ),

        "top1_total":
            N,

        "top1_agreement":
            float(
                top1_agreement_rate
            ),

        "top1_agreement_percent":
            float(
                100 *
                top1_agreement_rate
            ),

        "top5_set_matches":
            int(
                engine_top5_set_agreement
            ),

        "top5_set_agreement":
            float(
                top5_set_agreement_rate
            )
    },

    "numerical_equivalence": {

        "global_max_abs_logit_error":
            global_max_logit_error,

        "mean_sample_max_abs_logit_error":
            mean_sample_max_logit_error,

        "mean_abs_logit_error":
            mean_logit_error,

        "mean_rmse":
            mean_rmse,

        "global_max_probability_error":
            global_max_probability_error
    },

    "diagnostic_timing": {

        "pytorch_mean_ms":
            mean_pt_ms,

        "openvino_mean_ms":
            mean_ov_ms,

        "formal_benchmark":
            False,

        "formal_benchmark_stage":
            "L32"
    },

    "criteria":
        criteria,

    "all_equivalence_pass":
        bool(
            all_equivalence_pass
        ),

    "locked_test": {

        "loaded":
            False,

        "inference":
            False,

        "evaluation":
            False
    },

    "next_stage":
        "L24_CISLR_MP4_EXPECTED_VS_PREDICTED_DEMO"
}


save_json(

    summary,
    SUMMARY_JSON
)


print(
    "[SAVED]"
)

print(
    SUMMARY_JSON
)


# ==================================================================================================
# 23. HASH L23 ARTIFACTS
# ==================================================================================================

section(
    "23. HASH L23 ARTIFACTS"
)


artifact_paths = {

    "source_l21_checkpoint":
        L21_CHECKPOINT,

    "source_l22_xml":
        OPENVINO_XML,

    "source_l22_bin":
        OPENVINO_BIN,

    "validation_manifest":
        VAL_MANIFEST,

    "predictions_csv":
        PREDICTIONS_CSV,

    "verification_config":
        CONFIG_JSON,

    "verification_summary":
        SUMMARY_JSON
}


artifact_hashes = {}


for name, path in artifact_paths.items():

    digest = sha256_file(
        path
    )


    artifact_hashes[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest,

        "size_bytes":
            int(
                path.stat().st_size
            )
    }


    print(
        f"{name:<30}: "
        f"{digest}"
    )


save_json(

    artifact_hashes,
    HASH_JSON
)


print()
print(
    "[SAVED]"
)

print(
    HASH_JSON
)


# ==================================================================================================
# 24. CREATE L23 MASTER FREEZE
# ==================================================================================================

section(
    "24. CREATE L23 MASTER FREEZE"
)


master = {

    "stage":
        "L23",

    "status":
        (
            "COMPLETE_AND_FROZEN"
            if all_equivalence_pass
            else
            "COMPLETE_WITH_EQUIVALENCE_FAILURE"
        ),

    "name":
        "PyTorch OpenVINO real-validation verification",

    "source_models": {

        "pytorch_checkpoint":
            str(
                L21_CHECKPOINT
            ),

        "pytorch_sha256":
            actual_l21_sha,

        "openvino_xml":
            str(
                OPENVINO_XML
            ),

        "openvino_xml_sha256":
            actual_xml_sha,

        "openvino_bin":
            str(
                OPENVINO_BIN
            ),

        "openvino_bin_sha256":
            actual_bin_sha
    },

    "validation": {

        "manifest":
            str(
                VAL_MANIFEST
            ),

        "samples":
            N,

        "pytorch_correct":
            int(
                pytorch_correct
            ),

        "openvino_correct":
            int(
                openvino_correct
            ),

        "pytorch_accuracy_percent":
            float(
                100 *
                pytorch_accuracy
            ),

        "openvino_accuracy_percent":
            float(
                100 *
                openvino_accuracy
            )
    },

    "equivalence": {

        "top1_matches":
            int(
                engine_top1_agreement
            ),

        "top1_total":
            N,

        "top1_agreement_percent":
            float(
                100 *
                top1_agreement_rate
            ),

        "top5_set_agreement_percent":
            float(
                100 *
                top5_set_agreement_rate
            ),

        "global_max_abs_logit_error":
            global_max_logit_error,

        "global_max_probability_error":
            global_max_probability_error,

        "pass":
            bool(
                all_equivalence_pass
            )
    },

    "scientific_integrity": {

        "training":
            False,

        "retraining":
            False,

        "validation_tuning":
            False,

        "locked_test_manifest_loaded":
            False,

        "locked_test_video_loaded":
            False,

        "locked_test_inference":
            False,

        "locked_test_evaluation":
            False
    },

    "artifacts": {

        "predictions_csv":
            str(
                PREDICTIONS_CSV
            ),

        "config":
            str(
                CONFIG_JSON
            ),

        "summary":
            str(
                SUMMARY_JSON
            ),

        "hash_manifest":
            str(
                HASH_JSON
            )
    },

    "next_stage":
        "L24_CISLR_MP4_EXPECTED_VS_PREDICTED_DEMO"
}


save_json(

    master,
    MASTER_FREEZE
)


MASTER_SHA = sha256_file(
    MASTER_FREEZE
)


print(
    "[SAVED]"
)

print(
    MASTER_FREEZE
)


print()
print(
    "L23 MASTER SHA256:"
)

print(
    MASTER_SHA
)


# ==================================================================================================
# 25. FINAL REPORT
# ==================================================================================================

section(
    "L23 — PYTORCH <-> OPENVINO VERIFICATION COMPLETE"
)


print(
    "VALIDATION POPULATION"
)

print(
    f"  Samples                : {N}"
)

print(
    "  Locked test used       : NO"
)


print()
print(
    "RECOGNITION"
)

print(
    f"  PyTorch correct         : "
    f"{pytorch_correct}/{N}"
)

print(
    f"  PyTorch accuracy        : "
    f"{100*pytorch_accuracy:.3f}%"
)

print(
    f"  OpenVINO correct        : "
    f"{openvino_correct}/{N}"
)

print(
    f"  OpenVINO accuracy       : "
    f"{100*openvino_accuracy:.3f}%"
)


print()
print(
    "ENGINE EQUIVALENCE"
)

print(
    f"  Top-1 agreement         : "
    f"{engine_top1_agreement}/{N} "
    f"({100*top1_agreement_rate:.3f}%)"
)

print(
    f"  Top-5 set agreement     : "
    f"{engine_top5_set_agreement}/{N} "
    f"({100*top5_set_agreement_rate:.3f}%)"
)


print()
print(
    "NUMERICAL EQUIVALENCE"
)

print(
    "  Global max logit error :",
    f"{global_max_logit_error:.12e}"
)

print(
    "  Mean abs logit error   :",
    f"{mean_logit_error:.12e}"
)

print(
    "  Mean RMSE              :",
    f"{mean_rmse:.12e}"
)

print(
    "  Max probability error  :",
    f"{global_max_probability_error:.12e}"
)


print()
print(
    "EQUIVALENCE STATUS"
)

print(
    "  ",
    "PASS"
    if all_equivalence_pass
    else
    "FAIL"
)


print()
print(
    "SCIENTIFIC INTEGRITY"
)

print(
    "  Training               : NO"
)

print(
    "  Retraining             : NO"
)

print(
    "  Validation tuning      : NO"
)

print(
    "  Locked test loaded      : NO"
)

print(
    "  Locked test inference   : NO"
)

print(
    "  Locked test evaluation  : NO"
)


print()
print(
    "PREDICTIONS"
)

print(
    PREDICTIONS_CSV
)


print()
print(
    "MASTER FREEZE"
)

print(
    MASTER_FREEZE
)


print()
print(
    "MASTER SHA256"
)

print(
    MASTER_SHA
)


print()
print(
    "NEXT"
)

print(
    "L24 — CISLR MP4 EXPECTED-vs-PREDICTED OPENVINO DEMO"
)


print()
print("=" * 120)


# ==================================================================================================
# 26. CLEANUP
# ==================================================================================================

del model
del checkpoint
del state_dict
del compiled_model
del ov_model

gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


print()
print(
    "[PASS] L23 cleanup complete."
)


CISLR — L23 PYTORCH <-> OPENVINO REAL-VALIDATION VERIFICATION
Python       : 3.11.15
PyTorch      : 2.14.0+cu130
OpenVINO     : 2026.3.1-22476-759c5a6ab8c-releases/2026/3
Transformers : 4.57.6

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

Training              : DISABLED
Retraining            : DISABLED
Validation tuning     : DISABLED
Locked-test loading   : DISABLED
Locked-test inference : DISABLED
Locked-test evaluation: DISABLED

3. PATH CONFIGURATION
Validation manifest:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l18b_10class_dataset/manifests/l18b_validation_manifest.csv

PyTorch checkpoint:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l21_deployment_classifier_freeze/model/l21_frozen_deployment_videomae_10class.pt

OpenVINO XML:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l22_openvino_export/model/l22_cislr_videomae_10class.xml

L23 output:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LAN

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base-finetuned-kinetics and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters: 86,234,890

[PASS] Frozen L21 PyTorch model reconstructed.

14. LOAD FROZEN OPENVINO MODEL
Input shape : [1, 16, 3, 224, 224]
Output shape: [1, 10]
Device: CPU

[PASS] Frozen L22 OpenVINO model compiled.

15. BEGIN REAL VALIDATION EQUIVALENCE TEST
Samples: 10
IMPORTANT:
The SAME float32 tensor is supplied to PyTorch and OpenVINO.
No independent preprocessing path is used.

------------------------------------------------------------------------------------------------------------------------
[01/10] UID=0YIk1iTjDVY_1
Expected: 0 -> monday
PyTorch : 0 -> monday | conf=0.93529029
OpenVINO: 0 -> monday | conf=0.93529074
Correct PT/OV: True / True
Top-1 engine match: True
Top-5 exact order match: True
Top-5 set match: True
Max abs logit error: 7.152557373047e-06
Max probability error: 4.504187057774e-07

------------------------------------------------------------------------------------------------------------------------
[02/10] UID=jMWvOjMUjYk_1
Expected: 1 -> interest
PyTor